In [1]:
# NOTE that this needs to be started from a jupyter notebook that's within the IRAF27 environment.
import os.path
import os
import subprocess
import shutil
import sys
import glob
from cStringIO import StringIO
import itertools
import functools
import collections
from datetime import datetime
import cPickle as pickle

from astropy.io import fits
from astropy.table import Table, vstack, join
from astropy.modeling import fitting, models
from astropy.coordinates import SkyCoord, Angle
import astropy.units as u
from astropy.time import Time
from pyraf import iraf
iraf.set(stdimage="imt2048")
import matplotlib
%matplotlib qt5
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d
from scipy import stats
from scipy.special import gamma
from scipy.signal import lombscargle

In [2]:
BINARY_PATH = os.environ["THESIS"]
IMAGE_PATH = os.path.join(BINARY_PATH, "Modspec")
# Iraf tasks can have a maximum of 63 characters. So I don't want to work with absolute paths, just in case.
iraf.cd(IMAGE_PATH)

In [3]:
# Load in the packages we want
iraf.noao()
iraf.imred()
iraf.ccdred()
iraf.twodspec()
iraf.longslit()
iraf.apextract()
iraf.rv()
iraf.longslit.disp = 2
obsnights = [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

imred/:
 argus/         ctioslit/       hydra/          kpnocoude/      vtel/
 bias/          dtoi/           iids/           kpnoslit/
 ccdred/        echelle/        irred/          quadred/
 crutil/        generic/        irs/            specred/
ccdred/:
 badpiximage    ccdlist         combine         mkillumcor      setinstrument
 ccdgroups      ccdmask         darkcombine     mkillumflat     zerocombine
 ccdhedit       ccdproc         flatcombine     mkskycor
 ccdinstrument  ccdtest         mkfringecor     mkskyflat
twodspec/:
 apextract/     longslit/
longslit/:
 aidpars@       deredden        identify        sarith          specplot
 autoidentify   dopcor          illumination    scopy           specshift
 background     extinction      lcalib          sensfunc        splot
 bplot          fceval          lscombine       setairmass      standard
 calibrate      fitcoords       reidentify      setjd           transform
 demos          fluxcalib       response        sflip
apextr

In [ ]:

RAW_FOLDER = "Modspec_Raw"
TRIMMED_FOLDER = "Modspec_Trimmed"
# First copy the raw data before performing operations on it.
#iraf.cp(RAW_FOLDER, TRIMMED_FOLDER)
iraf.cd(TRIMMED_FOLDER)
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = True
# I just want to remove the overscan region and trim the images
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = True
iraf.ccdproc.trim = True
iraf.ccdproc.zerocor = False
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = False
iraf.ccdproc.illumcor = False
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.biassec = "[308:384,1:1700]"
iraf.ccdproc.trimsec = "[1:300,1:1700]"

iraf.ccdproc.interactive = True
iraf.ccdproc.order = 6

# Do this for running the whole sample
#for i in obsnights:
#    iraf.ccdproc(os.path.join(TRIMMED_PATH, "night{0:d}/night{0:d}.*.fit".format(i)))
iraf.ccdproc("night1/night1.0*.fit")

In [7]:
iraf.prows("night1/night1.001.fit", 100, 1600)

The image of the full chip shows a gradient across the chip over the spatial axis of around 20 counts. Hopefully this will be removed by the bias. The gradient also exists across the overscan region.

In [41]:
iraf.prows("night1/night1.001.fit", 100, 1600)

In [42]:
iraf.pcols("night1/night1.001.fit", 10, 290)

With the trimmed image, you basically see the spatial gradient as before. However, down the chip on the dispersion axis, 
there is very little gradient. Maybe of around 2 counts.

Now let's look at the differences between the bias frames of these objects.

In [6]:
ZEROED_FOLDER = "Modspec_Zeroproc"
iraf.cd(IMAGE_PATH)
shutil.copytree(TRIMMED_FOLDER, ZEROED_FOLDER)
iraf.cd(ZEROED_FOLDER)

NameError: name 'TRIMMED_FOLDER' is not defined

In [19]:
for i in obsnights:
    shutil.rmtree("night{0:d}/Biasdiffs/".format(i), ignore_errors=True)
    iraf.mkdir("night{0:d}/Biasdiffs/".format(i))
    with open(os.path.join(IMAGE_PATH, TRIMMED_FOLDER, "Night{0:d}_Biases.txt".format(i))) as biases:
        biaslist = biases.readlines()
        reference_index = 6
        ref_frame = biaslist[reference_index][:-1]
        ref_number = int(ref_frame[-7:-4])
        for j in xrange(len(biaslist)):
            bias_frame = biaslist[j][:-1]
            bias_number = int(bias_frame[-7:-4])
            iraf.imarith(bias_frame, "-", ref_frame, "night{0:d}/Biasdiffs/Biasdiff{1:d}{2:d}.fit".format(
                i, bias_number, ref_number)) 

I looked through the differences between the bias images for all of the nights. Some notable findings are:

* All nights have frames with transient diagonal structure in the bias images. It is not in phase between nights.

* Transient structure amplitude is small even on small scales. When running pcols over a very narrow column range to probe the amplitude of the structure, it was lost in the Poisson noise between the frames.

* On occasion, the diagonal structure can be irregular and fringy on certain frames. This level of fringiness is on the order of 5 counts. Less in other frames.

* Night 5 did not have frames with structure.

In [47]:
iraf.combine.combine = "average"
iraf.combine.reject = "minmax"
iraf.combine.scale = "none"
iraf.combine.nlow = 0
iraf.combine.nhigh = 1
iraf.combine.mclip = "yes"
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3

iraf.mkdir("Calibrations")
for i in obsnights:
    iraf.combine("Night{0:d}_Biases.txt".format(i), os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(i)))

<function pyraf.iraffunctions.wrapper>

In [20]:
shutil.rmtree(os.path.join("Calibrations", "Biasdiffs"), ignore_errors=True)
iraf.mkdir(os.path.join("Calibrations", "Biasdiffs"))
ref_night = 6
reference_bias = os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(ref_night))
for i in obsnights:
    current_bias = os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(i))
    iraf.imarith(current_bias, "-", reference_bias, os.path.join("Calibrations", "Biasdiffs", 
                                                                 "Biasdiffn{0:d}n{1:d}.fit".format(i, ref_night)))

In [ ]:
difflist = glob.glob(os.path.join(IMAGE_PATH, ZEROED_FOLDER, "Calibrations", "Biasdiffs", "Biasdiff*.fit"))
for diffimg in difflist:
    imgname = os.path.basename(diffimg)
    iraf.display(os.path.join("Calibrations", "Biasdiffs", imgname), 1, zscale=False, zrange=False, z1=-2, z2=2)
    print("Displaying {0}".format(imgname))
    wait = raw_input("Hit enter for next image.")

There is additional structure between nights on the 0.3 count level. It also slopes by 0.3 counts over the spatial axis. The dispersion axis seems stable and well-behaved.

In [19]:
for i in obsnights:
    with open(os.path.join(IMAGE_PATH, ZEROED_FOLDER, "Night{0:d}_Biases.txt".format(i))) as biases:
        biaslist = biases.readlines()
        night_bias = os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(i))
        for j in xrange(len(biaslist)):
            bias_frame = os.path.join("..", TRIMMED_FOLDER, biaslist[j][:-1])
            bias_number = int(bias_frame[-7:-4])
            try:
                os.remove(os.path.join(IMAGE_PATH, ZEROED_FOLDER, "night{0:d}".format(i), "Biasdiffs", 
                                       "Biasdiff{1:d}n{0:d}.fit".format(i, bias_number)))
            except OSError:
                pass
            iraf.imarith(bias_frame, "-", night_bias, os.path.join("night{0:d}".format(i), "Biasdiffs", 
                                   "Biasdiff{1:d}n{0:d}.fit".format(i, bias_number)))

In [ ]:
nightno = 14
difflist = glob.glob(os.path.join(IMAGE_PATH, ZEROED_FOLDER, "night{0:d}".format(nightno), "Biasdiffs", 
                                  "Biasdiff*n{0:d}.fit".format(nightno)))
for diffimg in difflist:
    imgname = os.path.basename(diffimg)
    iraf.display(os.path.join("night{0:d}".format(nightno), "Biasdiffs", imgname), 1, zscale=False, zrange=False, z1=-2, z2=2)
    print("Displaying {0}".format(imgname))
    wait = raw_input("Hit enter for next image.")

The fringes are still around. At this point, there doesn't seem to be an issue. I think at this point, there won't be an improvement in averaging things further.

In [32]:
biasdiffs = glob.glob(os.path.join(IMAGE_PATH, TRIMMED_FOLDER, "night*", "Biasdiffs"))
copylocs = map(lambda x: x.replace(TRIMMED_FOLDER, ZEROED_FOLDER), biasdiffs)
for src, dst in zip(biasdiffs, copylocs):
    shutil.copytree(src, dst)

In [17]:
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = True
# Only do the zero correction
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = False
iraf.ccdproc.trim = False
iraf.ccdproc.zerocor = True
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = False
iraf.ccdproc.illumcor = False
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.interactive = True

# Do this for running the whole sample
#for i in obsnights:
#    iraf.ccdproc(os.path.join("night{0:d}".format(i), "night{0:d}.*.fit".format(i)), 
#                              zero=os.path.join("Calibrations", "Night{0:d}_Zero.fit".format(i))
iraf.ccdproc("night1/night1.0*.fit")

# Flat Fielding

Now it's time to look at the flat field exposures. Here I'll note the ratios between the fields.

In [55]:
FLATFIELD_FOLDER = "Modspec_Flatproc"
iraf.cd(IMAGE_PATH)
#shutil.copytree(os.path.join(IMAGE_PATH, ZEROED_FOLDER), os.path.join(IMAGE_PATH, FLATFIELD_FOLDER))
iraf.cd(FLATFIELD_FOLDER)

In [55]:
for i in obsnights:
    shutil.rmtree("night{0:d}/Flatratios/".format(i), ignore_errors=True)
    iraf.mkdir("night{0:d}/Flatratios/".format(i))
    with open(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "Night{0:d}_Flats.txt".format(i))) as flats:
        flatlist = flats.readlines()
        reference_index = 6
        ref_frame = flatlist[reference_index][:-1]
        ref_number = int(ref_frame[-7:-4])
        # Need to scale the flats. I want to use imstat for this.
        ref_imstat_output = iraf.imstat(ref_frame+"[1:300,1:1200]", Stdout=1, fields="image,npix,mode")
        ref_imstat_dict = dict(zip(ref_imstat_output[0][1:].split(), ref_imstat_output[1].split()))
        ref_mode = float(ref_imstat_dict["MODE"])
        for j in xrange(len(flatlist)):
            flat_frame = flatlist[j][:-1]
            flat_number = int(flat_frame[-7:-4])
            flat_imstat_output = iraf.imstat(flat_frame+"[1:300,1:1200]", Stdout=1, fields="image,npix,mode")
            flat_imstat_dict = dict(zip(flat_imstat_output[0][1:].split(), flat_imstat_output[1].split()))
            flat_mode = float(flat_imstat_dict["MODE"])
            scaled_frame = flat_frame.replace(".{0:03d}.".format(flat_number), ".s{0:03d}.".format(flat_number))
            iraf.imarith(flat_frame, "*", ref_mode / flat_mode, scaled_frame)
            iraf.imarith(flat_frame, "/", ref_frame, "night{0:d}/Flatratios/Flatratio{1:d}{2:d}.fit".format(
                i, flat_number, ref_number)) 

In [ ]:
nightno = 5
ratiolist = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "night{0:d}".format(nightno), "Flatratios", 
                                  "Flatratio*.fit".format(nightno)))
for ratioimg in ratiolist:
    imgname = os.path.basename(ratioimg)
    iraf.display(os.path.join("night{0:d}".format(nightno), "Flatratios", imgname), 1, zscale=False, zrange=False, z1=0.9, 
                 z2=1.1)
    print("Displaying {0}".format(imgname))
    wait = raw_input("Hit enter for next image.")

In [76]:
nightno = 14
ratiolist = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "night{0:d}".format(nightno), "Flatratios", 
                                  "Flatratio*.fit".format(nightno)))
current_path = iraf.pwd(Stdout=1)[0]
for ratioimg in ratiolist:
    imgname = os.path.relpath(ratioimg, current_path)
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.pcols(imgname, 1, 300, append=append, wy1=0.6, wy2=1.4)

There is a definite trend over many nights where the flat field lamp varies in brightness and temperature. The slope of the curve definitely correlates with the overall brightness of the lamp. When the lamp is brighter, it slopes up, when the lamp is fainter, it slopes down compared to a standard exposure.

As a result, it's preferable to fit the response function first, and then combine the flat fields.

In [54]:
iraf.twodspec()
iraf.longslit()

twodspec/:
 apextract/     longslit/
longslit/:
 aidpars@       deredden        identify        sarith          specplot
 autoidentify   dopcor          illumination    scopy           specshift
 background     extinction      lcalib          sensfunc        splot
 bplot          fceval          lscombine       setairmass      standard
 calibrate      fitcoords       reidentify      setjd           transform
 demos          fluxcalib       response        sflip


In [11]:
iraf.response.interactive = True
iraf.response.order = 11
iraf.response.low_reject = 3
iraf.response.high_reject = 3

for i in obsnights:
    with open(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "Night{0:d}_Flats.txt".format(i))) as flats:
        flatlist = flats.readlines()
        for flatname in flatlist:
            flatname = flatname[:-1]
            flat_number = int(flatname[-7:-4])
            corrected_flat = flatname.replace(".{0:03d}.".format(flat_number), ".n{0:03d}.".format(flat_number))
            iraf.response(flatname, flatname, corrected_flat)

NameError: name 'FLATFIELD_FOLDER' is not defined

The flats should all be corrected and normalized now, and stored in .n???.fit files. Check to make sure that the flat fields are well-behaved.

In [12]:
nightno = 3
normed = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "night{0:d}".format(nightno), 
                                "night{0:d}.n*.fit".format(nightno)))
current_path = iraf.pwd(Stdout=1)[0]
for norm in normed:
    if norm is normed[0]:
        append=False
    else:
        append=True
    iraf.pcols(norm, 1, 300, append=append, wy1=0.98, wy2=1.02)

The flat fields are fairly uniform and well-behaved up to pixel 1200, after which they get to be pretty ratty. Be wary of using the flatfield past that. Before that, the flatfields seem to be uniform down to 0.5%. Now let's combine these normalized flatfields.

In [ ]:
combine.reject = "avsigclip"
combine.scale = "mode"
combine.nlow = 1
combine.nhigh = 1
combine.nkeep = 1
combine.lsigma = 3
combine.hsigma = 3
combine.statsec="[1:300,1:1200]"

for i in obsnights:
    iraf.combine("Night{0:d}_Flats.txt".format(i), output=os.path.join("Calibrations", "Night{0:d}_Flat.fit".format(i)))
        

In [13]:
shutil.rmtree(os.path.join("Calibrations", "Flatratios"), ignore_errors=True)
iraf.mkdir(os.path.join("Calibrations", "Flatratios"))
ref_night = 6
reference_flat = os.path.join("Calibrations", "Night{0:d}_Flat.fit".format(ref_night))
for i in obsnights:
    current_flat = os.path.join("Calibrations", "Night{0:d}_Flat.fit".format(i))
    iraf.imarith(current_flat, "/", reference_flat, os.path.join("Calibrations", "Flatratios", 
                                                                 "Flatration{0:d}n{1:d}.fit".format(i, ref_night)))

In [ ]:
ratiolist = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "Calibrations", "Flatratios", "Flatratio*.fit"))
current_path = iraf.pwd(Stdout=1)[0]
for ratioimg in ratiolist:
    imgname = os.path.relpath(ratioimg, current_path)
    iraf.display(imgname, 1, zscale=False, zrange=False, z1=0.95, z2=1.05)
    print("Displaying {0}".format(imgname))
    wait = raw_input("Hit enter for next image.")
    

Nightly combined flats seem to differ from each other on a level of 0.1%, which is extremely small. I think we should just combine all flats into a master flat. Additionally, the differences in structure seen on the chip seemed to be small compared to the flat noise.

In [ ]:
iraf.combine(os.path.join("Calibrations", "Night*_Flat.fit"), output=os.path.join("Calibrations", "Master_Flat.fit"))

In [20]:
iraf.display(os.path.join("Calibrations", "Master_Flat.fit"), 1, zscale=False, zrange=False, z1=0.9, z2=1.1)

z1=0.9 z2=1.1


In [ ]:
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = True
# Only do the zero correction
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = False
iraf.ccdproc.trim = False
iraf.ccdproc.zerocor = False
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = True
iraf.ccdproc.illumcor = False
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.interactive = True

# Do this for running the whole sample
for i in obsnights:
    iraf.ccdproc("@Night{0:d}_Ne.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    iraf.ccdproc("@Night{0:d}_Xe.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    iraf.ccdproc("@Night{0:d}_Ar.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    try:
        iraf.ccdproc("@Night{0:d}_Twilight.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    except iraf.IrafError:
        pass
    iraf.ccdproc("@Night{0:d}_Objects.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))

# Illumination


Get the illumination correction handled correctly now that the flat field is complete.

In [4]:
ILLUM_FOLDER = "Modspec_Illumproc"
iraf.cd(IMAGE_PATH)
#shutil.copytree(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER), os.path.join(IMAGE_PATH, ILLUM_FOLDER))
iraf.cd(ILLUM_FOLDER)

In order to do the illumination corrections, we have to look at the twilight exposures. These should now have been flatfield-corrected.

In [5]:
nightno = 13
scale_value = 1000.0
with open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Twilight.txt".format(nightno))) as twilights:
    twilist = twilights.readlines()
    for twi in twilist:
        twiname = twi[:-1]
        twi_number = int(twiname[-7:-4])
        twi_imstat_output = iraf.imstat(twiname+"[10:290,100:1300]", Stdout=1, fields="image,npix,midpt,mode")
        twi_imstat_dict = dict(zip(twi_imstat_output[0][1:].split(), twi_imstat_output[1].split()))
        twi_mode = float(twi_imstat_dict["MIDPT"])
        scaled_frame = twiname.replace(".{0:03d}.".format(twi_number), ".s{0:03d}.".format(twi_number))
        try:
            os.remove(os.path.join(IMAGE_PATH, ILLUM_FOLDER, scaled_frame))
        except OSError:
            pass
        iraf.imarith(twiname, "*", scale_value / twi_mode, scaled_frame)
        print(twi_mode)
        if twi is twilist[0]:
            append = False
        else:
            append = True
        iraf.prows(scaled_frame, 100, 1300, append=append)

1436.0
437.0


Slope changes down the chip. So we'll have to correct them piece by piece.

In [134]:
for i in obsnights:
    try:
        twilights = open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Twilight.txt".format(i))) 
    except IOError:
        pass
    else:
        twilist = twilights.readlines()
        reference_index = -1
        ref_name = twilist[reference_index][:-1]
        ref_number = int(ref_name[-7:-4])
        scaled_ref = ref_name.replace(".{0:03d}.".format(ref_number), ".s{0:03d}.".format(ref_number))
        for twi in twilist:
            twiname = twi[:-1]
            twi_number = int(twiname[-7:-4])
            scaled_frame = twiname.replace(".{0:03d}.".format(twi_number), ".s{0:03d}.".format(twi_number))
            try:
                os.remove("night{0:d}/Flatratios/Twiratio{1:d}{2:d}.fit".format(i, twi_number, ref_number))
            except OSError:
                pass
            iraf.imarith(scaled_frame, "/", scaled_ref, "night{0:d}/Flatratios/Twiratio{1:d}{2:d}.fit".format(
                i, twi_number, ref_number)) 

In [37]:
nightno = 4
ratiolist = glob.glob(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "night{0:d}".format(nightno), "Flatratios", "Twiratio*.fit"))
currentpath = iraf.pwd(Stdout=1)[0]
for ratioimg in ratiolist:
    imgname = os.path.relpath(os.path.realpath(ratioimg), currentpath)
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.prows(imgname, 400, 800, append=append, wy1=0.95, wy2=1.05)
    

In [7]:
nightno = 11
ratiolist = glob.glob(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "night{0:d}".format(nightno), "Flatratios", 
                                  "Twiratio*.fit".format(nightno)))
currentpath = iraf.pwd(Stdout=1)[0]
for ratioimg in ratiolist:
    imgname = os.path.relpath(ratioimg, currentpath)
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.pcols(imgname, 220, 290, append=append, wy1=0.8, wy2=1.2)

Strange that the behavior of the twilights seems to be different along the dispersion axis. This may mean that the illumination changes somehow. It's strange. Hopefully it doesn't indicate that I messed up with the dome.

Twilight flats seem to vary by around 10% along the dispersion axis.

Along the spatial axis, twilights don't seem to vary at all. There are some differenes in the scaling, but generally they are consistently flat within a night.

In [19]:
iraf.combine.reject = "avsigclip"
iraf.combine.scale = "median"
iraf.combine.weight = "median"
iraf.combine.blank = 1
iraf.combine.nlow = 1
iraf.combine.nhigh = 1
iraf.combine.nkeep = 1
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3
iraf.combine.statsec="[10:290,1:1200]"

# Make twilight frames for each night
for i in obsnights:
    os.remove(os.path.join("Calibrations", "Night{0:d}_Twilight.fit".format(i)))
    iraf.combine("Night{0:d}_Twilight.txt".format(i), 
                 output=os.path.join("Calibrations", "Night{0:d}_Twilight.txt".format(i)))
        

Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have different dimensions
Input images have differe

In [47]:
nightno = 12
twitargs = open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Twilight.txt".format(i)))
twilist = twitargs.readlines()
ratiolist = map(lambda x: x.replace(".{0}.".format(x[-8:-5]), ".f{0}.".format(x[-8:-5])), twilist)
for ratioimg in ratiolist[1:2]:
    imgname = ratioimg[:-1]
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.pcols(imgname, 130, 145, append=False, wy1=1.0, wy2=1.5)

In [43]:
# Now I want to do this for the Kepler target exposures to see if it matches.
for i in obsnights:
    keptargs = open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_KIC_Objects.txt".format(i)))
    for kep in keptargs:
        kepname = kep[:-1]
        kep_number = int(kepname[-7:-4])
        left_name = kepname+"[1:150,1:1700]"
        right_name = kepname+"[151:300,1:1700]"
        scaled_frame = kepname.replace(".{0:03d}.".format(kep_number), ".f{0:03d}.".format(kep_number))
        iraf.imarith(right_name, "/", left_name, scaled_frame) 
    keptargs.close()

In [52]:
nightno = 12
keptargs = open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_KIC_Objects.txt".format(i)))
keplist = keptargs.readlines()
ratiolist = map(lambda x: x.replace(".{0}.".format(x[-8:-5]), ".f{0}.".format(x[-8:-5])), keplist)
for ratioimg in ratiolist[5:6]:
    imgname = ratioimg[:-1]
    if ratioimg is ratiolist[0]:
        append=False
    else:
        append=True
    iraf.pcols(imgname, 100, 145, append=False, wy1=1.0, wy2=1.5)

This didn't work out well. There's way too much noise in these observations. I'll try co-adding all of the Kepler observations and then seeing if that helps tamp down the noise.

In [59]:
iraf.combine.reject = "avsigclip"
iraf.combine.scale = "median"
iraf.combine.nlow = 1
iraf.combine.nhigh = 1
iraf.combine.nkeep = 1
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3
iraf.combine.statsec="[1:75,1:1200]"

for i in obsnights:
    os.remove(os.path.join("Calibrations", "Night{0:d}_Sky.fit".format(i)))
    iraf.combine("@Night{0:d}_KIC_Objects.txt".format(i), output=os.path.join("Calibrations", "Night{0:d}_Sky.fit".format(i)))
        

In [61]:
for i in obsnights:
    sky_image = os.path.join("Calibrations", "Night{0:d}_Sky.fit".format(i))
    left_name = sky_image+"[1:150,1:1700]"
    right_name = sky_image+"[151:300,1:1700]"
    scaled_frame = sky_image.replace("Sky", "Skyratio")
    iraf.imarith(right_name, "/", left_name, scaled_frame) 

In [ ]:
nightno = 4
iraf.pcols(os.path.join("Calibrations", "Night{0:d}_Skyratio.fit".format(nightno)), )

In [58]:
flatratios = glob.glob(os.path.join(IMAGE_PATH, FLATFIELD_FOLDER, "night*", "Flatratios"))
copylocs = map(lambda x: x.replace(FLATFIELD_FOLDER, ILLUM_FOLDER), flatratios)
for src, dst in zip(flatratios, copylocs):
    shutil.copytree(src, dst)

* Show that twilight exposures are the same long the dispersion axis.
* Compare twilights to sky values.
* Combine twilights.

In [44]:
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = True
# Only do the zero correction
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = False
iraf.ccdproc.trim = False
iraf.ccdproc.zerocor = False
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = True
iraf.ccdproc.illumcor = False
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.interactive = True

# Do this for running the whole sample
for i in obsnights:
    iraf.ccdproc("@Night{0:d}_Ne.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    iraf.ccdproc("@Night{0:d}_Xe.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    iraf.ccdproc("@Night{0:d}_Ar.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    try:
        iraf.ccdproc("@Night{0:d}_Twilight.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))
    except iraf.IrafError:
        pass
    iraf.ccdproc("@Night{0:d}_Objects.txt".format(i),flat=os.path.join("Calibrations", "Master_Flat.fit"))

night14/night14.s073.fit


In [139]:
# Create text files for object files.
for i in obsnights[2:3]:
    with open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Objects.txt".format(i))) as targets, \
         open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_KIC_Objects.txt".format(i)), "w") as kics, \
         open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Standards.txt".format(i)), "w") as standards, \
         open(os.path.join(IMAGE_PATH, ILLUM_FOLDER, "Night{0:d}_Arcs.txt".format(i)), "w") as arcs:
        for frame in targets:
            objpath = os.path.join(IMAGE_PATH, ILLUM_FOLDER, frame[:-1])
            targethdu = fits.open(objpath)
            objname = targethdu[0].header["OBJECT"]
            exptime = targethdu[0].header["EXPTIME"]
            targethdu.close()
            if objname.endswith("Arc"):
                targetfile = arcs
            elif objname.startswith("HD") or objname.startswith("BD") or objname.startswith("HIP"):
                targetfile = standards
            elif objname.startswith("KIC"):
                targetfile = kics
            else:
                print("Don't know file: {0}".format(objpath))
                continue
            targetfile.write(frame)

NameError: name 'ILLUM_FOLDER' is not defined

In [6]:
iraf.combine.reject = "avsigclip"
iraf.combine.scale = "median"
iraf.combine.weight = "median"
iraf.combine.blank = 1
iraf.combine.nlow = 1
iraf.combine.nhigh = 1
iraf.combine.nkeep = 1
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3
iraf.combine.statsec="[10:290,1:1200]"


os.remove(os.path.join("Calibrations", "Master_Twilight.fit"))
iraf.combine("@Twilights.txt", output=os.path.join("Calibrations", "Master_Twilight.fit"))
        

In [20]:
iraf.illum.interact = True
iraf.illum.nbins = 9
iraf.illum.low_reject = 3
iraf.illum.high_reject = 3
iraf.illum.order = 5

#os.remove(os.path.join("Calibrations", "Illum.fit"))
iraf.illum(os.path.join("Calibrations", "Master_Twilight.fit"), os.path.join("Calibrations", "Illum.fit"))

Determine illumination interactively for Calibrations/Master_Twilight.fit (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 1 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 2 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 3 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 4 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 5 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 6 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 7 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 8 (yes): Determine illumination interactively for Calibrations/Master_Twilight.fit at bin 9 (yes): 

The last bin looked kinda weird. But overall the twilight corrections seemed to be good!

In [24]:
iraf.ccdproc.ccdtype = ""
iraf.ccdproc.noproc = False
iraf.ccdproc.fixpix = False
iraf.ccdproc.overscan = False
iraf.ccdproc.trim = False
iraf.ccdproc.zerocor = False
iraf.ccdproc.darkcor = False
iraf.ccdproc.flatcor = False
# Only do the illumination correction.
iraf.ccdproc.illumcor = True
iraf.ccdproc.fringec = False
iraf.ccdproc.readcor = False
iraf.ccdproc.scancor = False

iraf.ccdproc.interactive = True

# Do this for running the whole sample
for i in obsnights:
    iraf.ccdproc("@Night{0:d}_Ne.txt".format(i),illum=os.path.join("Calibrations", "Illum.fit"))
    iraf.ccdproc("@Night{0:d}_Xe.txt".format(i),illum=os.path.join("Calibrations", "Illum.fit"))
    iraf.ccdproc("@Night{0:d}_Ar.txt".format(i),illum=os.path.join("Calibrations", "Illum.fit"))
    iraf.ccdproc("@Night{0:d}_Objects.txt".format(i),illum=os.path.join("Calibrations", "Illum.fit"))

# Calibration

In [4]:
CALIB_FOLDER = "Modspec_Calibration"
iraf.cd(IMAGE_PATH)
#shutil.copytree(os.path.join(IMAGE_PATH, ILLUM_FOLDER), os.path.join(IMAGE_PATH, CALIB_FOLDER))
iraf.cd(CALIB_FOLDER)

## Extract all of the spectra

In [185]:
iraf.apall.interactive = True
iraf.apall.find = True
iraf.apall.recenter = True
iraf.apall.resize = False
iraf.apall.edit = True
iraf.apall.trace = True
iraf.apall.extract = True
iraf.apall.review = True

iraf.apall.line = 148
iraf.apall.nsum = 10
iraf.apall.width = 24
iraf.apall.lower = -12
iraf.apall.upper = 12
iraf.apall.resize = False

iraf.apall.b_sample = "-100:-30,30:100"
iraf.apall.b_naver = -100
iraf.apall.b_funct = "chebyshev"
iraf.apall.b_order = 1
iraf.apall.b_high_rej = 3
iraf.apall.b_niter = 5
iraf.apall.b_grow = 1

iraf.apall.t_nsum = 10
iraf.apall.t_step = 10
iraf.apall.t_funct = "spline3"
iraf.apall.t_order = 2
iraf.apall.t_niter = 1

iraf.background = "fit"
iraf.apall.weights = "none"
iraf.apall.clean = False
iraf.apall.format = "multispec"
iraf.apall.extras = True

In [5]:
# These four files need to be existing in order to generate the other files:
# Night12_Standards.txt
# Night12_KIC_Objects.txt
# Night12_Standards_Arcs.txt
# Night12_KIC_Objects_Arcs.txt

def compact_standard(filename):
    '''Compactify a filename.'''
    compact = os.path.splitext(os.path.splitext(os.path.basename(filename))[0])[0].replace(".", "").replace("night","n")
    return compact

# These are for simple file naming. Just format and go!
# Examples of the file types are given above the template
obj_types = ("Standards", "KIC_Objects")
arctypes = ["ne", "xe", "ar"]
# raw_target_template.format(12, objtypes[0]) -> Night12_Standards.txt
raw_target_template = "Night{0:d}_{1}.txt"
# combined_target_template.format(12, objtypes[0]) -> Night12_Standards_Combined.txt
combined_target_template = "Night{0:d}_{1}_Combined.txt"
# extracted_target_template.format(12, objtypes[0]) -> Night12_Standards_Extracted.txt
extracted_target_template = "Night{0:d}_{1}_Extracted.txt"
# calibration_template.format(12, objtypes[0]) -> Night12_Standards_Calib.txt
calibrated_target_template = "Night{0:d}_{1}_Calib.txt"
# calibration_template.format(12, objtypes[0], arctypes[2].capitalize()) -> Night12_Standards_Ar.txt
calibration_template = "Night{0:d}_{1}_{2}.txt"
# repeat_calibration_template.format(12, objtypes[0], arctypes[2].capitalize()) -> Night12_Standards_Repeat_Ar.txt
repeat_calibration_template = "Night{0:d}_{1}_Repeat_{2}.txt"
# subtracted_calibration_template.format(12, objtypes[0], arctypes[2].capitalize() -> Night12_Standards_Ar_Subtracted.txt)
subtracted_calibration_template = "Night{0:d}_{1}_{2}_Subtracted.txt"
# fullspec_template.format(12, objtypes[0]) -> Night12_Standards_Fullspec.txt
fullspec_template = "Night{0:d}_{1}_Fullspec.txt"
# arc_template.format(12, objtypes[0]) -> Night12_Standards_Arcs.txt
arc_template = "Night{0:d}_{1}_Arcs.txt"
# extracted_arc_template.format(12, objtypes[0]) -> Night12_Standards_Arcs_Extracted.txt
extracted_arc_template = "Night{0:d}_{1}_Arcs_Extracted.txt"
# fxcor_flexure_template.format(12, objtypes[0]) -> Night12_Standards_Cor_Base.txt
fxcor_flexure_template = "Night{0:d}_{1}_Cor_Base.txt"
# arc_multitrace_template.format(12, objtypes[0], 121) -> Night12_Standards_Arc121.txt
arc_multitrace_template = "Night{0:d}_{1}_Arc{2}.txt"
# fxcor_multitrace_template.format(12, objtypes[0], 121) -> Night12_Standards_Arc121_FXcor.txt
fxcor_multitrace_template = "Night{0:d}_{1}_Arc{2}_FXcor.txt"
# target_cor_template.format(12, objtypes[0], compact_standard("night12/night12.c121.ms.fits").upper()) -> Night12_Standards_Cor_N12C121.txt
target_cor_template = "Night{0:d}_{1}_Cor_{2}.txt"
# cleaned_target_template.format(12, obj_types[0]) -> Night12_Standards_Cleaned.txt
cleaned_target_template = "Night{0:d}_{1}_Cleaned.txt"
# linearized_target_template.format(12, obj_types[0]) -> Night12_Standard_Linear.txt
linearized_target_template = "Night{0:d}_{1}_Linear.txt"

In [27]:
iraf.imcombine.scale

'none'

In [25]:
for n in obsnights[0:1]:
    img_count = 1
    for targ in obj_types:
        raw_target_filelist = raw_target_template.format(n, targ)
        combined_target_filelist = combined_target_template.format(n, targ)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, raw_target_filelist), "r") as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, combined_target_filelist), "w") as newfile:
                for imageline in oldfile:
                    images = imageline[:-1].split(" ")
                    if len(images) == 1:
                        newimage = images[0]
                    elif len(images) > 1:
                        newimage = images[0].replace(images[0][-7:-4],"d{0:02d}".format(img_count))
                        try:
                            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newimage))
                        except OSError:
                            pass
                        iraf.imcombine(",".join(images), newimage)
                        img_count = img_count + 1
                    newfile.write(newimage+"\n")
                        
                
        extracted_target_filelist = extracted_target_template.format(n, targ)
        # Make the filenames for the extracted objects
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, combined_target_filelist), 'r') as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist), 'w') as newfile:
                for oldname in oldfile:
                    newname = oldname.replace(".fit", ".ms.fits")
                    newfile.write(newname)
#                    os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
        iraf.apall("@"+combined_target_filelist, output="@"+extracted_target_filelist, intera="yes")


Oct 16 13:31: IMCOMBINE
  combine = average, scale = none, zero = none, weight = none
  blank = 0.
                Images 
  night1/night1.075.fit
  night1/night1.076.fit

  Output image = night1/night1.d01.fit, ncombine = 2

Oct 16 13:31: IMCOMBINE
  combine = average, scale = none, zero = none, weight = none
  blank = 0.
                Images 
  night1/night1.113.fit
  night1/night1.114.fit

  Output image = night1/night1.d02.fit, ncombine = 2

Oct 16 13:31: IMCOMBINE
  combine = average, scale = none, zero = none, weight = none
  blank = 0.
                Images 
  night1/night1.122.fit
  night1/night1.123.fit

  Output image = night1/night1.d03.fit, ncombine = 2

Oct 16 13:31: IMCOMBINE
  combine = average, scale = none, zero = none, weight = none
  blank = 0.
                Images 
  night1/night1.127.fit
  night1/night1.128.fit

  Output image = night1/night1.d04.fit, ncombine = 2

Oct 16 13:31: IMCOMBINE
  combine = average, scale = none, zero = none, weight = none
  blank =

Recenter apertures for night1/night1.072?Edit apertures for night1/night1.072?

     aperture = 1  beam = 1  center = 149.41  low = -3.36  upper = 3.03


Trace apertures for night1/night1.072?Fit traced positions for night1/night1.072 interactively?Fit curve to aperture 1 of night1/night1.072 interactivelyWrite apertures for night1/night1.072 to databaseExtract aperture spectra for night1/night1.072?Review extracted spectra from night1/night1.072?Clobber existing output image night1/night1.072.ms?

Oct 16 13:31: EXTRACT - Output spectrum night1/night1.072.ms already exists


...

Oct 16 13:32: EXTRACT - Output spectrum night1/night1.109.ms already exists


Recenter apertures for night1/night1.110?Edit apertures for night1/night1.110?

     aperture = 1  beam = 1  center = 147.02  low = -12.00  upper = 12.00


Trace apertures for night1/night1.110?Fit traced positions for night1/night1.110 interactively?Fit curve to aperture 1 of night1/night1.110 interactivelyWrite apertures for night1/night1.110 to databaseExtract aperture spectra for night1/night1.110?Review extracted spectra from night1/night1.110?Clobber existing output image night1/night1.110.ms?

Oct 16 13:32: EXTRACT - Output spectrum night1/night1.110.ms already exists


## Clean Spectra

In [41]:
# Clean spectra of cosmic rays by running through splot by hand.
placeholder_file = "moveme.fits"
print "Since splot cannot directly save to files, make sure to save the output"
print 'to {0}. This function will automatically place that file into'.format(placeholder_file)
print "the correct location. This will ensure the user doesn't have to constantly"
print "keep typing in locations."
for n in obsnights:
    for targ in obj_types:
        extracted_target_files = extracted_target_template.format(n, targ)
        cleaned_target_files = cleaned_target_template.format(n, targ)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_files), 'r') as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, cleaned_target_files), 'w') as newfile:
                for oldname in oldfile:
                    nightstr = oldname[:oldname.index(os.sep)]+"."
                    newname = oldname.replace(nightstr, nightstr+"x")
                    newfile.write(newname)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_files)) as extracted, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, cleaned_target_files)) as cleaned:
                for extractedfile, cleanedfile in itertools.izip(extracted, cleaned):
                    if not os.path.isfile(cleanedfile[:-1]):
                        iraf.splot(extractedfile[:-1], 1, 1)
                        shutil.move(os.path.join(IMAGE_PATH, CALIB_FOLDER, placeholder_file), 
                                    os.path.join(IMAGE_PATH, CALIB_FOLDER, cleanedfile[:-1]))
        

Since splot cannot directly save to files, make sure to save the output
to moveme.fits. This function will automatically place that file into
the correct location. This will ensure the user doesn't have to constantly
keep typing in locations.


IOError: [Errno 2] No such file or directory: '/home/regulus/simonian/Binaries/Modspec/Modspec_Calibration/moveme.fits'

In [113]:
iraf.splot.boxsize=""

## Wavelength-Calibrate Spectra

In [7]:
xc=(0, 0, 0)
nc=(0, 109/255.0, 219/255.0)
ac=(219/255.0, 209/255.0, 0)
fc = (182/255.0, 219/255.0, 255/255.0)

In [7]:
# I want to put all the spectra on the same wavelength scale.
wavelength_scale = (4500, 6000, 1)
ws=wavelength_scale

In [26]:
# Generate the Calibration Spectra
iraf.combine.combine = "average"
iraf.combine.scale = "mode"
iraf.combine.weight = "mode"
iraf.combine.reject = "avsigclip"
iraf.combine.mclip = "yes"
iraf.combine.nkeep = 1
iraf.combine.lsigma = 3
iraf.combine.hsigma = 3
iraf.combine.statsec = "[1:300,900:1100]"
for n in obsnights:
    for spec in arctypes:
        combofile = "Night{0:d}_{1}.txt".format(n, spec.capitalize())
        outputfile = os.path.join("Calibrations", "Night{0:d}_{1}.fit".format(n, spec.capitalize()))
        try:
            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, outputfile))
        except OSError:
            pass
        iraf.combine("@"+combofile, outputfile)

In [19]:
# Extract calibration spectra using object and standard traces.
for n in obsnights:
    for targclass in obj_types:
        raw_target_filelist = combined_target_template.format(n, targclass)
        extracted_target_filelist = cleaned_target_template.format(n, targclass)
        for spec in arctypes:
            extracted_calib_filelist = calibration_template.format(n, targclass, spec.capitalize())
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist), 'r') as oldfile:
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), 'w') as newfile:
                    for oldname in oldfile:
                        # turn night12.212.ms.fits to night12.ar212.ms.fits
                        nightname = oldname[:oldname.index(os.sep)]+"."
                        newname = oldname.replace(nightname, nightname+spec)
                        newfile.write(newname)
                        try:
                            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                            print "Removed " + newname
                        except OSError:
                            pass
            # Since apall can't deal with a single input, I'll loop through the name files manually with python instead 
            # of just creating a file with the input spectrum repeating.
            apall_repeat_file = repeat_calibration_template.format(n, targclass, spec.capitalize())
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), 'r') as oldfile:
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, apall_repeat_file), 'w') as newfile:
                    for oldname in oldfile:
                        # Just write the master Calibration file over and over.
                        newname = os.path.join("Calibrations", "Night{0:d}_{1}.fit\n".format(n, spec.capitalize()))
                        newfile.write(newname)
            iraf.apall("@"+apall_repeat_file, out="@"+extracted_calib_filelist, ref="@"+raw_target_filelist, recen=False, 
                        trace=False, back="none", intera=False)
            print "Things Extracted."
            # Subtract the continuum from the lines.
            subtracted_calib_filelist = subtracted_calibration_template.format(n, targclass, spec.capitalize())
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), 'r') as oldfile:
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, subtracted_calib_filelist), 'w') as newfile:
                    for oldname in oldfile:
                        # turn night12.ar212.ms.fits to night12.sar212.ms.fits
                        newname = oldname.replace(spec, "s"+spec)
                        newfile.write(newname)
                        try:
                            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                            print "Removed " + newname
                        except OSError:
                            pass
            iraf.continuum.func = "chebyshev"
            iraf.continuum.order = 15
            iraf.continuum.high_rej = 3
            iraf.continuum.low_rej = 0
            # Continuum isn't happy with empty files. So ignore this if it's empty.
            try:
                iraf.continuum("@"+extracted_calib_filelist, "@"+subtracted_calib_filelist, intera="no")
            except iraf.IrafError:
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), 'r') as infile:
                    contents = infile.readlines()
                    if not contents:
                        pass
                    else:
                        raise

            # Now reidentify the lines
            iraf.reidentify(os.path.join("calib_test", "{0}spec".format(spec)), "@"+subtracted_calib_filelist, intera="no")
        
# Combine the line identifications and refit the wavelength solution.
        fullspec_filelist = fullspec_template.format(n, targclass)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, 
                               subtracted_calibration_template.format(n, targclass, arctypes[2].capitalize())), "r") as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist), "w") as newfile:
            for oldname in oldfile:
                # turn night12.ar212.ms.fits to night12.full212.ms.fits
                newname = oldname.replace("sar", "full")
                shutil.copy(os.path.join(IMAGE_PATH, CALIB_FOLDER, oldname[:-1]), 
                            os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                newfile.write(newname)
        # Read in all of the features
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist), "r") as fullspecs:
            for fullimg in fullspecs:
                fulldb, ext = os.path.splitext(os.path.join("database", "id"+fullimg))
                full_spec_table = []
                for spec in ["ne", "ar", "xe"]:
                    specimg = fullimg.replace("full", "s"+spec)
                    specdb, ext = os.path.splitext(os.path.join("database", "id"+specimg))
                    # Read in the entry.
                    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, specdb)) as specdata:
                        spec_fullfile = specdata.read()
                    spec_features = spec_fullfile[spec_fullfile.rindex("begin"):]
                    spec_length_line_start = spec_features.index("features")
                    spec_length_line_end = spec_features.index("\n", spec_length_line_start)
                    spec_numlines = int(spec_features[spec_length_line_start:spec_length_line_end].split("\t")[1])
                    spec_table_start = spec_length_line_end+1
                    # Subtract two because there is a trailing tab before "function"
                    spec_table_end = spec_features.index("function")-2
                    spec_table = spec_features[spec_table_start:spec_table_end].split("\n")
                    full_spec_table = full_spec_table + spec_table
                # Now combine them
                full_numlines = len(full_spec_table)
                full_feat_table = Table.read(full_spec_table, format="ascii.fixed_width_no_header", 
                                            names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"), 
                                            col_starts=(0, 15, 26, 37, 43, 45), col_ends=(14, 25, 36, 40, 44, 46))
                full_feat_table["Count"] = np.arange(len(full_feat_table))
                full_feat_table.sort("Pixel")
                sorted_table = [full_spec_table[i] for i in full_feat_table["Count"]]
                new_feature_table = "\n".join(sorted_table)

                # Now let's piece together the new file. First make the time comment.
                a = datetime.now()
                comment_line = "# " + a.strftime("%a %H:%M:%S %d-%b-%Y") + "\n"
                # Then make the header:
                spec_head_start = 0
                # Note that this includes the leading tab character in the header, not as part of the "feature" line.
                spec_head_end = spec_length_line_start
                spec_header = spec_features[spec_head_start:spec_head_end]
                full_header = spec_header.replace("s"+spec, "full")
                # Now the feature line will be added on.
                full_feature_line = "features\t{0:d}\n".format(full_numlines)
                # Lastly we want the footer, which doesn't actually contain any useful information, but we will include.
                footer = spec_features[spec_table_end:]
                # Now add them all together!
                fullfile = comment_line + full_header + full_feature_line + new_feature_table + footer
        
                # Write the result to a file.
                with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fulldb), 'a') as fulldata:
                    fulldata.write(fullfile)
# Apply the wavelength solution to the full spectra.
        calibrated_target_filelist = calibrated_target_template.format(n, targclass)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist), "r") as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_target_filelist), "w") as newfile:
            for oldname in oldfile:
                # turn night12.x212.ms.fits to night12.c212.ms.fits
                newname = oldname.replace("x", "c")
                newfile.write(newname)
        # I don't know why this isn't working. It says that the objects in the references list aren't real reference spectra.
        # Just set all of the spectra manually.
        # iraf.refspec("@"+extracted_target_filelist, references="@"+fullspec_filelist, confirm=True, select="match")
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist)) as fullspec, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist)) as stand:
                for ref, obj in zip(fullspec, stand):
                    print obj[:-1]
                    iraf.hedit(obj[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_target_filelist)) as caltarg:
            for targ in caltarg:
                try:
                    os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, targ[:-1]))
                except OSError:
                    pass
        iraf.dispcor("@"+extracted_target_filelist, "@"+calibrated_target_filelist, linearize=False)
    
# Extract standard arc spectra
    raw_standard_file = combined_target_template.format(n, obj_types[0])
    standard_arcfile = arc_template.format(n, obj_types[0])
    extracted_standard_arcfile = extracted_arc_template.format(n, obj_types[0])
    print (standard_arcfile, raw_standard_file)
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standard_arcfile), 'r') as oldfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_standard_arcfile), 'w') as newfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, raw_standard_file), 'r') as companionfile:
        for oldname, compname in zip(oldfile, companionfile):
            standnum = compname[-8:-5]
            arcnum = oldname[-8:-5]
            newname = oldname.replace(".fit", ".ms.fits").replace(arcnum, arcnum+"t"+standnum)
            newfile.write(newname)
            try:
                os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                print "Removed " + newname
            except OSError:
                pass
    iraf.apall("@"+standard_arcfile, out="@"+extracted_standard_arcfile, 
               ref="@"+raw_standard_file, recen=False, trace=False, back="none", intera=False)
# Cross-correlate extracted arc spectra with Argon trace to get flexure correction.
    iraf.fxcor.continuum = "both"
    iraf.fxcor.filter = "both"
    iraf.fxcor.pixcorr = "yes"
    iraf.fxcor.function = "gaussian"
    iraf.fxcor.observatory = "kpno"
    
    iraf.continpars.c_inter = True
    iraf.continpars.order = 20
    iraf.continpars.low_rej = 0
    iraf.continpars.high_rej = 2
    iraf.continpars.nitera = 10
    iraf.continpars.grow = 1
    
    iraf.filtpars.f_type = "square"
    iraf.filtpars.cuton = 30
    iraf.filtpars.cutoff = 1000
    
    fxcor_flexure_output = fxcor_flexure_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_standard_arcfile), 'r') as oldfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_flexure_output), 'w') as newfile:
        for oldname in oldfile:
            newname = os.path.splitext(os.path.splitext(oldname)[0])[0]+"\n"
            newfile.write(newname)
            
    standard_calibration_argon = calibration_template.format(n, obj_types[0], arctypes[2].capitalize())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_standard_arcfile)) as arcs, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standard_calibration_argon)) as calibs, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_flexure_output)) as corfile:
            for arc, ar, cor in zip(arcs, calibs, corfile):
                if arc[-12:-9] != ar[-12:-9]:
                    print "{0} does not correctly trace night{1}.{2}.ms.fits".format(arc[:-1], n, ar[-12:-9])
                iraf.fxcor(arc[:-1], ar[:-1], out=cor[:-1], interact="no")

# Apply flexure correction.
    standard_file = calibrated_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_flexure_output)) as exarc_file, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standard_file)) as standards:
        shiftnums = []
        for arcbase, standimage in zip(exarc_file, standards):
            shiftfile = arcbase[:-1] + ".txt"
            shift_table = Table.read(shiftfile, format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                             header_start=13, guess=False, 
                             names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                                    'VOBS', 'VREL', 'VHELIO', 'VERR'])
            shift = shift_table["SHIFT"][-1]
            
            # This is just to make sure that the standards and arcs match up. It's possible some weird things go on.
            objlabel = iraf.hedit(standimage[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip()
            arclabel = shift_table["OBJECT"][-1]
            if objlabel + "_Arc" != arclabel:
                print "{0} does not match arc {1}".format(standimage[:-1], shift_table["IMAGE"][-1])
                print "{0} is not the arc of {1}".format(arclabel, objlabel)
                raise "F this S"

            iraf.specshift(standimage[:-1], shift)
            print "Corrected " + standimage[:-1]
    
# Get trace from (???) for Kepler arc spectra and Argon calibration.
    raw_kic_file = combined_target_template.format(n, obj_types[1])
    extracted_kic_file = extracted_target_template.format(n, obj_types[1])
    kic_arcfile = arc_template.format(n, obj_types[1])
    extracted_kic_arcfile = extracted_arc_template.format(n, obj_types[1])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_arcfile), 'r') as arcs:
        total_shifts = []
        for arc in arcs:
            arcnum = arc[-8:-5]
            output_traces = arc_multitrace_template.format(n, obj_types[1], arcnum)
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, raw_kic_file), 'r') as oldfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces), 'w') as newfile:
                for oldname in oldfile:
                    kicnum = oldname[-8:-5]
                    newname = oldname.replace(".fit", ".ms.fits").replace(kicnum, arcnum+"t"+kicnum)
                    newfile.write(newname)
                    try:
                        os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                        print "Removed " + newname
                    except OSError:
                        pass
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, raw_kic_file), 'r') as kics, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces), 'r') as outputs:
                    for kic, out in zip(kics, outputs):
                        iraf.apall(arc[:-1], out=out[:-1], ref=kic[:-1], recen=False, trace=False, back="none", 
                                   intera=False)
# Measure flexure throughout the night. (Output plots)
            fxcor_base = fxcor_multitrace_template.format(n, obj_types[1], arcnum)
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces), 'r') as oldfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_base), 'w') as newfile:
                for oldname in oldfile:
                    # wname = os.path.splitext(os.path.splitext(oldname)[0])[0]+"\n"
                    newfile.write(newname)
            iraf.fxcor.continuum = "both"
            iraf.fxcor.filter = "both"
            iraf.fxcor.pixcorr = "yes"
            iraf.fxcor.function = "gaussian"
            iraf.fxcor.observatory = "kpno"
    
            iraf.continpars.c_inter = True
            iraf.continpars.order = 20
            iraf.continpars.low_rej = 0
            iraf.continpars.high_rej = 2
            iraf.continpars.nitera = 10
            iraf.continpars.grow = 1
    
            iraf.filtpars.f_type = "square"
            iraf.filtpars.cuton = 30
            iraf.filtpars.cutoff = 1000
    
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fxcor_base)) as corfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, output_traces)) as arcfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, 
                                   calibration_template.format(n, obj_types[1], arctypes[2].capitalize())), 'r') as tempfile:
                arcshifts = []
                for corbase, trace, template in zip(corfile, arcfile, tempfile):
                    iraf.fxcor(trace[:-1], template[:-1], out=corbase[:-1], interact="no")
                               
                # Measure scatter in flexure
                    shiftfile = corbase[:-1] + ".txt"
                    shift_table = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, shiftfile), 
                                             format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                                             header_start=13, guess=False, 
                                             names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 
                                                    'TDR', 'VOBS', 'VREL', 'VHELIO', 'VERR'])
                    shift = shift_table["SHIFT"][-1]
                    arcshifts.append(shift)
    
            total_shifts.append(arcshifts)
    shift_array = np.array(total_shifts)
    flexures = np.mean(shift_array, axis=1)
# Interpolate and apply flexure correction to targets 
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_arcfile)) as arcs:
        arctimes = []
        for arc in arcs:
            hdulist = fits.open(arc[:-1])
            arctimes.append(hdulist[0].header["JD"])
            hdulist.close()
    times = np.array(arctimes)
    zerotime = times[0]
    relative_jd = (times - zerotime)*24
    flexure_fit = np.polyfit(relative_jd, flexures, 2)
    flexfunc = np.poly1d(flexure_fit)
    
    calibrated_kic_file = calibrated_target_template.format(n, obj_types[1])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_kic_file)) as kics:
        for kicobj in kics:
            hdulist = fits.open(kicobj[:-1])
            try:
                flexcorr = flexfunc((hdulist[0].header["JD"]-zerotime)*24)
            except ValueError:
                # I accidentally observed the object before the arc. So manually set the flexure to that measured by the arc.
                if kicobj[:-1] == "night7/night7.c073.ms.fits":
                    print "Does fit care about being evaluated outside?"
                    
            hdulist.close()
            
            iraf.specshift(kicobj[:-1], flexcorr)
            print "Corrected " + kicobj[:-1]
    
    xvals = np.linspace(0, 12, 100)
    flexes = flexfunc(xvals)

# Now linearize them all.
    for targ in obj_types:
        linear_target_filelist = linearized_target_template.format(n, targ)
        calibrated_target_filelist = calibrated_target_template.format(n, targ)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_target_filelist), "r") as oldfile, \
             open(os.path.join(IMAGE_PATH, CALIB_FOLDER, linear_target_filelist), "w") as newfile:
                for oldname in oldfile:
                    # turn night12.c212.ms.fits to night12.l212.ms.fits
                    newname = oldname.replace("c", "l")
                    newfile.write(newname)
                    try:
                        os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                    except OSError:
                        pass
        iraf.dispcor("@"+calibrated_target_filelist, "@"+linear_target_filelist, linear=True, w1=ws[0], w2=ws[1], 
                     dw=ws[2], log="yes")

Removed night1/night1.nex072.ms.fits

Removed night1/night1.nexd01.ms.fits

Removed night1/night1.nex077.ms.fits

Removed night1/night1.nexd02.ms.fits

Removed night1/night1.nex115.ms.fits

Removed night1/night1.nex118.ms.fits

Removed night1/night1.nex119.ms.fits

Removed night1/night1.nexd03.ms.fits

Removed night1/night1.nex124.ms.fits

Removed night1/night1.nexd04.ms.fits

Removed night1/night1.nexd05.ms.fits

Things Extracted.
Removed night1/night1.snex072.ms.fits

Removed night1/night1.snexd01.ms.fits

Removed night1/night1.snex077.ms.fits

Removed night1/night1.snexd02.ms.fits

Removed night1/night1.snex115.ms.fits

Removed night1/night1.snex118.ms.fits

Removed night1/night1.snex119.ms.fits

Removed night1/night1.snexd03.ms.fits

Removed night1/night1.snex124.ms.fits

Removed night1/night1.snexd04.ms.fits

Removed night1/night1.snexd05.ms.fits

Removed night1/night1.xex072.ms.fits

Removed night1/night1.xexd01.ms.fits

Removed night1/night1.xex077.ms.fits

Removed night1/night1

Things Extracted.
Removed night1/night1.sxex081.ms.fits

Removed night1/night1.sxex082.ms.fits

Removed night1/night1.sxex083.ms.fits

Removed night1/night1.sxex084.ms.fits

Removed night1/night1.sxex085.ms.fits

Removed night1/night1.sxex086.ms.fits

Removed night1/night1.sxex087.ms.fits

Removed night1/night1.sxex089.ms.fits

Removed night1/night1.sxex090.ms.fits

Removed night1/night1.sxex091.ms.fits

Removed night1/night1.sxex092.ms.fits

Removed night1/night1.sxex093.ms.fits

Removed night1/night1.sxex094.ms.fits

Removed night1/night1.sxex095.ms.fits

Removed night1/night1.sxex097.ms.fits

Removed night1/night1.sxexd06.ms.fits

Removed night1/night1.sxex100.ms.fits

Removed night1/night1.sxex101.ms.fits

Removed night1/night1.sxex102.ms.fits

Removed night1/night1.sxex103.ms.fits

Removed night1/night1.sxex106.ms.fits

Removed night1/night1.sxex107.ms.fits

Removed night1/night1.sxex108.ms.fits

Removed night1/night1.sxex109.ms.fits

Removed night1/night1.sxex110.ms.fits

Removed

night1/night1.x089.ms.fits: REFSPEC1 = 'night1/night1.fullx089.ms.fits 1.'
night1/night1.c089.ms.fits: ap = 1, w1 = 6062.565, w2 = 4348.436, dw =  -1.0089, nw = 1700, log = yes
night1/night1.x090.ms.fits: REFSPEC1 = 'night1/night1.fullx090.ms.fits 1.'
night1/night1.c090.ms.fits: ap = 1, w1 =  6062.57, w2 = 4348.443, dw =  -1.0089, nw = 1700, log = yes
night1/night1.x091.ms.fits: REFSPEC1 = 'night1/night1.fullx091.ms.fits 1.'
night1/night1.c091.ms.fits: ap = 1, w1 = 6062.565, w2 = 4348.446, dw =  -1.0089, nw = 1700, log = yes
night1/night1.x092.ms.fits: REFSPEC1 = 'night1/night1.fullx092.ms.fits 1.'
night1/night1.c092.ms.fits: ap = 1, w1 = 6062.564, w2 = 4348.429, dw = -1.00891, nw = 1700, log = yes
night1/night1.x093.ms.fits: REFSPEC1 = 'night1/night1.fullx093.ms.fits 1.'
night1/night1.c093.ms.fits: ap = 1, w1 = 6062.564, w2 = 4348.446, dw =  -1.0089, nw = 1700, log = yes
night1/night1.x094.ms.fits: REFSPEC1 = 'night1/night1.fullx094.ms.fits 1.'
night1/night1.c094.ms.fits: ap = 1, w1 =

Removed night1/night1.111t081.ms.fits

Removed night1/night1.111t082.ms.fits

Removed night1/night1.111t083.ms.fits

Removed night1/night1.111t084.ms.fits

Removed night1/night1.111t085.ms.fits

Removed night1/night1.111t086.ms.fits

Removed night1/night1.111t087.ms.fits

Removed night1/night1.111t089.ms.fits

Removed night1/night1.111t090.ms.fits

Removed night1/night1.111t091.ms.fits

Removed night1/night1.111t092.ms.fits

Removed night1/night1.111t093.ms.fits

Removed night1/night1.111t094.ms.fits

Removed night1/night1.111t095.ms.fits

Removed night1/night1.111t097.ms.fits

Removed night1/night1.111td06.ms.fits

Removed night1/night1.111t100.ms.fits

Removed night1/night1.111t101.ms.fits

Removed night1/night1.111t102.ms.fits

Removed night1/night1.111t103.ms.fits

Removed night1/night1.111t106.ms.fits

Removed night1/night1.111t107.ms.fits

Removed night1/night1.111t108.ms.fits

Removed night1/night1.111t109.ms.fits

Removed night1/night1.111t110.ms.fits

Corrected night1/night1.c

Removed night3/night3.nex086.ms.fits

Removed night3/night3.nex087.ms.fits

Removed night3/night3.nex090.ms.fits

Removed night3/night3.nexd01.ms.fits

Removed night3/night3.nex095.ms.fits

Removed night3/night3.nex096.ms.fits

Removed night3/night3.nex100.ms.fits

Removed night3/night3.nex102.ms.fits

Removed night3/night3.nex103.ms.fits

Removed night3/night3.nex106.ms.fits

Removed night3/night3.nex107.ms.fits

Removed night3/night3.nex111.ms.fits

Removed night3/night3.nex112.ms.fits

Removed night3/night3.nex115.ms.fits

Removed night3/night3.nexd02.ms.fits

Removed night3/night3.nex120.ms.fits

Removed night3/night3.nex121.ms.fits

Removed night3/night3.nex124.ms.fits

Removed night3/night3.nex125.ms.fits

Removed night3/night3.nex176.ms.fits

Removed night3/night3.nex177.ms.fits

Removed night3/night3.nex179.ms.fits

Removed night3/night3.nex182.ms.fits

Removed night3/night3.nex183.ms.fits

Removed night3/night3.nex186.ms.fits

Removed night3/night3.nex187.ms.fits

Removed nigh

...

Removed night14/night14.139t134.ms.fits

Removed night14/night14.139t135.ms.fits

Removed night14/night14.139t136.ms.fits

Removed night14/night14.139t137.ms.fits

Removed night14/night14.139t138.ms.fits

Removed night14/night14.139t140.ms.fits

Removed night14/night14.139t141.ms.fits

Removed night14/night14.139t142.ms.fits

Removed night14/night14.139t143.ms.fits

Removed night14/night14.139t144.ms.fits

Removed night14/night14.139t146.ms.fits

Removed night14/night14.139t147.ms.fits

Removed night14/night14.139t148.ms.fits

Removed night14/night14.139t149.ms.fits

Removed night14/night14.139t150.ms.fits

Removed night14/night14.139t151.ms.fits

Removed night14/night14.139t153.ms.fits

Removed night14/night14.139t154.ms.fits

Removed night14/night14.139t155.ms.fits

Removed night14/night14.139t156.ms.fits

Removed night14/night14.139t157.ms.fits

Removed night14/night14.139t159.ms.fits

Removed night14/night14.139t160.ms.fits

Removed night14/night14.139t161.ms.fits

Removed night14/

Removed night14/night14.165t134.ms.fits

Removed night14/night14.165t135.ms.fits

Removed night14/night14.165t136.ms.fits

Removed night14/night14.165t137.ms.fits

Removed night14/night14.165t138.ms.fits

Removed night14/night14.165t140.ms.fits

Removed night14/night14.165t141.ms.fits

Removed night14/night14.165t142.ms.fits

Removed night14/night14.165t143.ms.fits

Removed night14/night14.165t144.ms.fits

Removed night14/night14.165t146.ms.fits

Removed night14/night14.165t147.ms.fits

Removed night14/night14.165t148.ms.fits

Removed night14/night14.165t149.ms.fits

Removed night14/night14.165t150.ms.fits

Removed night14/night14.165t151.ms.fits

Removed night14/night14.165t153.ms.fits

Removed night14/night14.165t154.ms.fits

Removed night14/night14.165t155.ms.fits

Removed night14/night14.165t156.ms.fits

Removed night14/night14.165t157.ms.fits

Removed night14/night14.165t159.ms.fits

Removed night14/night14.165t160.ms.fits

Removed night14/night14.165t161.ms.fits

Removed night14/

Removed night14/night14.191t134.ms.fits

Removed night14/night14.191t135.ms.fits

Removed night14/night14.191t136.ms.fits

Removed night14/night14.191t137.ms.fits

Removed night14/night14.191t138.ms.fits

Removed night14/night14.191t140.ms.fits

Removed night14/night14.191t141.ms.fits

Removed night14/night14.191t142.ms.fits

Removed night14/night14.191t143.ms.fits

Removed night14/night14.191t144.ms.fits

Removed night14/night14.191t146.ms.fits

Removed night14/night14.191t147.ms.fits

Removed night14/night14.191t148.ms.fits

Removed night14/night14.191t149.ms.fits

Removed night14/night14.191t150.ms.fits

Removed night14/night14.191t151.ms.fits

Removed night14/night14.191t153.ms.fits

Removed night14/night14.191t154.ms.fits

Removed night14/night14.191t155.ms.fits

Removed night14/night14.191t156.ms.fits

Removed night14/night14.191t157.ms.fits

Removed night14/night14.191t159.ms.fits

Removed night14/night14.191t160.ms.fits

Removed night14/night14.191t161.ms.fits

Removed night14/

night14/night14.l094.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c095.ms.fits: Resampling using current coordinate system
night14/night14.l095.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c098.ms.fits: Resampling using current coordinate system
night14/night14.l098.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c099.ms.fits: Resampling using current coordinate system
night14/night14.l099.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c102.ms.fits: Resampling using current coordinate system
night14/night14.l102.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c103.ms.fits: Resampling using current coordinate system
night14/night14.l103.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/nig

night14/night14.c149.ms.fits: Resampling using current coordinate system
night14/night14.l149.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c150.ms.fits: Resampling using current coordinate system
night14/night14.l150.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c151.ms.fits: Resampling using current coordinate system
night14/night14.l151.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c153.ms.fits: Resampling using current coordinate system
night14/night14.l153.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c154.ms.fits: Resampling using current coordinate system
night14/night14.l154.ms.fits: ap = 1, w1 =    4500., w2 =    6000., dw =       1., nw = 1501, log = yes
night14/night14.c155.ms.fits: Resampling using current coordinate system
night14/night14.l155.ms.fits: ap = 1, w1 =

In [50]:


xe_solution = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "night{0:d}xe.txt".format(Calib_Night)), 
                         format="ascii.no_header", names=("wv", "int"))
ne_solution = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "night{0:d}ne.txt".format(Calib_Night)), 
                         format="ascii.no_header", names=("wv", "int"))
ar_solution = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "night{0:d}ar.txt".format(Calib_Night)), 
                         format="ascii.no_header", names=("wv", "int"))
full_solution = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "night{0:d}full.txt".format(Calib_Night)),
                          format="ascii.no_header", names=("wv", "int"))
second_full = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "night{0:d}".format(Calib_Night), 
                                      "night{0:d}.cfull234.txt".format(Calib_Night)), 
                         format="ascii.no_header", names=("wv", "int"))

xe_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "xenon.feat"), 
                     format="ascii.fixed_width", header_start=2, data_end=23, guess=False, col_starts=(2, 11, 22, 33), 
                     col_ends=(10, 21, 32, 43))
ne_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "neon.feat"), 
                     format="ascii.fixed_width", header_start=2, data_end=25, guess=False, col_starts=(2, 11, 22, 33), 
                     col_ends=(10, 21, 32, 43))
ar_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "argon.feat"), 
                     format="ascii.fixed_width", header_start=2, data_end=42, guess=False, col_starts=(2, 11, 22, 33), 
                     col_ends=(10, 21, 32, 43))
full_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "calib_test", "all.feat"),
                       format="ascii.fixed_width", header_start=2, data_end=84, guess=False, col_starts=(2, 11, 22, 33),
                      col_ends=(10, 21, 32, 43))
full2_feat = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, "night12", "full2.feat"),
                       format="ascii.fixed_width", header_start=2, data_end=73, guess=False, col_starts=(2, 11, 22, 33),
                      col_ends=(10, 21, 32, 43))

full_xefeat = full_feat[np.array([v in xe_feat["User"] for v in full_feat["User"]])]
full_nefeat = full_feat[np.array([v in ne_feat["User"] for v in full_feat["User"]])]
full_arfeat = full_feat[np.array([v in ar_feat["User"] for v in full_feat["User"]])]

full2_xefeat = full2_feat[np.array([v in xe_feat["User"] for v in full2_feat["User"]])]
full2_nefeat = full2_feat[np.array([v in ne_feat["User"] for v in full2_feat["User"]])]
full2_arfeat = full2_feat[np.array([v in ar_feat["User"] for v in full2_feat["User"]])]


pixels = np.arange(1700)+1

NameError: name 'Calib_Night' is not defined

In [11]:
allfeatures = full_feat
init_model = models.Linear1D(slope=-1, intercept=6063)
fitter = fitting.LevMarLSQFitter()
disp = fitter(init_model, allfeatures["Pixel"], allfeatures["User"])

linear = disp(pixels)
xe_nonlinear = xe_solution["wv"] - linear
ne_nonlinear = ne_solution["wv"] - linear
ar_nonlinear = ar_solution["wv"] - linear

xefeat_nonlinear = full_xefeat["User"] - disp(full_xefeat["Pixel"])
nefeat_nonlinear = full_nefeat["User"] - disp(full_nefeat["Pixel"])
arfeat_nonlinear = full_arfeat["User"] - disp(full_arfeat["Pixel"])

plt.plot(pixels, xe_nonlinear, color=xc, marker=".", label="Xenon")
plt.plot(pixels, ne_nonlinear, color=nc, marker=".", label="Neon")
plt.plot(pixels, ar_nonlinear, color=ac, marker=".", label="Argon")
plt.legend(loc="upper left")
plt.plot(xe_feat["Pixel"], xefeat_nonlinear, color=xc, marker="d", ls="")
plt.plot(ne_feat["Pixel"], nefeat_nonlinear, color=nc, marker="o", ls="")
plt.plot(ar_feat["Pixel"], arfeat_nonlinear, color=ac, marker="s", ls="")
plt.xlabel("Pixel")
plt.ylabel("Non-linear Part")

In [21]:
full_nonlinear = full_solution["wv"] - linear

xefeat_nonlinear = full_xefeat["User"] - disp(full_xefeat["Pixel"])
nefeat_nonlinear = full_nefeat["User"] - disp(full_nefeat["Pixel"])
arfeat_nonlinear = full_arfeat["User"] - disp(full_arfeat["Pixel"])

plt.plot(pixels, full_nonlinear, color=fc, marker=".", label="Joint")
plt.plot(full_xefeat["Pixel"], xefeat_nonlinear, color=xc, marker="d", ls="", label="Xenon")
plt.plot(full_nefeat["Pixel"], nefeat_nonlinear, color=nc, marker="o", ls="", label="Neon")
plt.plot(full_arfeat["Pixel"], arfeat_nonlinear, color=ac, marker="s", ls="", label="Argon")
plt.legend(loc="upper left")
plt.xlabel("Pixel")
plt.ylabel("Non-linear Part")

In [35]:
full2_nonlinear = second_full["wv"] - linear

xefeat_nonlinear = full2_xefeat["User"] - disp(full2_xefeat["Pixel"])
nefeat_nonlinear = full2_nefeat["User"] - disp(full2_nefeat["Pixel"])
arfeat_nonlinear = full2_arfeat["User"] - disp(full2_arfeat["Pixel"])

plt.plot(pixels, full_nonlinear, color=fc, marker=".", label="Joint")
plt.plot(full2_xefeat["Pixel"], xefeat_nonlinear, color=xc, marker="d", ls="", label="Xenon")
plt.plot(full2_nefeat["Pixel"], nefeat_nonlinear, color=nc, marker="o", ls="", label="Neon")
plt.plot(full2_arfeat["Pixel"], arfeat_nonlinear, color=ac, marker="s", ls="", label="Argon")
#plt.legend(loc="upper left")
plt.xlabel("Pixel")
plt.ylabel("Non-linear Part")

In [37]:
xefeat_residual = -full_xefeat["Residual"]
nefeat_residual = -full_nefeat["Residual"]
arfeat_residual = -full_arfeat["Residual"]

plt.plot(full_xefeat["Pixel"], xefeat_residual, color=xc, marker="d", ls="", label="Xenon")
plt.plot(full_nefeat["Pixel"], nefeat_residual, color=nc, marker="o", ls="", label="Neon")
plt.plot(full_arfeat["Pixel"], arfeat_residual, color=ac, marker="s", ls="", label="Argon")
plt.legend(loc="upper right")
plt.plot([pixels[0], pixels[-1]], [0, 0], color=fc, ls="--")
plt.xlabel("Pixel")
plt.ylabel("Residual (User - Fit)")

In [39]:
xefeat_residual = -full2_xefeat["Residual"]
nefeat_residual = -full2_nefeat["Residual"]
arfeat_residual = -full2_arfeat["Residual"]

plt.plot(full2_xefeat["Pixel"], xefeat_residual, color=xc, marker="d", ls="", label="Xenon")
plt.plot(full2_nefeat["Pixel"], nefeat_residual, color=nc, marker="o", ls="", label="Neon")
plt.plot(full2_arfeat["Pixel"], arfeat_residual, color=ac, marker="s", ls="", label="Argon")
plt.legend(loc="upper right")
plt.plot([pixels[0], pixels[-1]], [0, 0], color=fc, ls="--")
plt.xlabel("Pixel")
plt.ylabel("Residual (User - Fit)")
plt.title("Wavelength Solution 234")

plt.show()

In [79]:
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, "Night12_Arcs.txt"),"r") as arcs:
    arcfiles = arcs.readlines()
    slopes = np.zeros(shape=len(arcfiles))
    intercepts = np.zeros(shape=len(arcfiles))
    times = np.zeros(shape=len(arcfiles))
    airmasses = np.zeros(shape=len(arcfiles))
    for i, arcfile in enumerate(arcfiles):
        base, ext = os.path.splitext(arcfile[:-1])
        dbfile = "".join([os.path.join(IMAGE_PATH, CALIB_FOLDER, "database", "id"), base])
        # First look for the end of the data.
        with open(dbfile, "r") as database:
            fullfile = database.read()
        lastentry = fullfile[fullfile.rindex("begin"):]
        features_index = lastentry.index("features")
        tablelength = int(lastentry[features_index+8:features_index+lastentry[features_index:].index("\n")])
        feat_table = Table.read(lastentry, format="ascii.fixed_width", names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"),
                               data_start=6, data_end=6+tablelength, col_starts=(0, 15, 26, 37, 43, 45), 
                                col_ends=(14, 25, 36, 40, 44, 46))
        joined_feats = join(feat_table, full2_arfeat, join_type="inner", table_names=("STD", "ARC"), keys="User")
        ideal_fit = models.Linear1D(slope=1, intercept=0)
        fitter = fitting.LevMarLSQFitter()
        actual_fit = fitter(ideal_fit, joined_feats["Pixel_STD"], joined_feats["Pixel_ARC"])
        slopes[i] = actual_fit.slope.value
        intercepts[i] = actual_fit.intercept.value
        
        # Now let's get the time of observation and the airmass
        fitsfile = os.path.join(IMAGE_PATH, CALIB_FOLDER, arcfile[:-1])
        hdulist = fits.open(fitsfile)
        times[i] = hdulist[0].header["JD"]
        airmasses[i] = hdulist[0].header["AIRMASS"]

In [94]:
timediff = (times - times[0])*24
plt.plot(timediff, slopes, 'k+')
plt.plot([timediff[0], timediff[-1]], [1, 1], 'k--')
plt.xlabel("Time since first exposure")
plt.ylabel("Slope")
plt.title("Dispersion difference")

In [95]:
plt.plot(airmasses-1, slopes, 'k+')
plt.plot([0, 1], [1, 1], 'k--')
plt.xlabel("Airmass")
plt.ylabel("Slope")
plt.title("Dispersion difference")

In [27]:
matplotlib.interactive(False)
# Now let's go through the images and generate plots of the residuals.
# I'll want to save them in a subdirectory in the plots folder.
calib_residual_path = os.path.join(os.environ["THESIS"], "plots", "calib_residuals")
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_standards)) as fullspecs:
    for fullimg in fullspecs:
        # These are the image names
        arimg = fullimg.replace("full", "sar")
        neimg = fullimg.replace("full", "sne")
        xeimg = fullimg.replace("full", "sxe")
        # Now get the database names.
        ardb, ext = os.path.splitext(os.path.join("database", "id"+arimg))
        nedb, ext = os.path.splitext(os.path.join("database", "id"+neimg))
        xedb, ext = os.path.splitext(os.path.join("database", "id"+xeimg))
        # Now the main feature file.
        featfile = os.path.splitext(os.path.splitext(fullimg)[0])[0] + ".feat"
                # First read in the argon entry.
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, ardb)) as ardata:
            ar_fullfile = ardata.read()
        ar_features = ar_fullfile[ar_fullfile.rindex("begin"):]
        ar_length_line_start = ar_features.index("features")
        ar_length_line_end = ar_features.index("\n", ar_length_line_start)
        ar_numlines = int(ar_features[ar_length_line_start:ar_length_line_end].split("\t")[1])
        ar_table_start = ar_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        ar_table_end = ar_features.index("function")-2
        ar_table = ar_features[ar_table_start:ar_table_end].split("\n")
        ar_feat_table = Table.read(ar_table, format="ascii.fixed_width_no_header", 
                                   names=("Pixel", "Fit", "User", "Fwidth", "Wt"), 
                                   col_starts=(0, 15, 26, 37, 43), col_ends=(14, 25, 36, 40, 44))
        # Now read in the neon and xenon entries
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, nedb)) as nedata:
            ne_fullfile = nedata.read()
        ne_features = ne_fullfile[ne_fullfile.rindex("begin"):]
        ne_length_line_start = ne_features.index("features")
        ne_length_line_end = ne_features.index("\n", ne_length_line_start)
        ne_numlines = int(ne_features[ne_length_line_start:ne_length_line_end].split("\t")[1])
        ne_table_start = ne_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        ne_table_end = ne_features.index("function")-2
        ne_table = ne_features[ne_table_start:ne_table_end].split("\n")
        ne_feat_table = Table.read(ne_table, format="ascii.fixed_width_no_header", 
                                   names=("Pixel", "Fit", "User", "Fwidth", "Wt"), 
                                   col_starts=(0, 15, 26, 37, 43), col_ends=(14, 25, 36, 40, 44))
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, xedb)) as xedata:
            xe_fullfile = xedata.read()
        xe_features = xe_fullfile[xe_fullfile.rindex("begin"):]
        xe_length_line_start = xe_features.index("features")
        xe_length_line_end = xe_features.index("\n", xe_length_line_start)
        xe_numlines = int(xe_features[xe_length_line_start:xe_length_line_end].split("\t")[1])
        xe_table_start = xe_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        xe_table_end = xe_features.index("function")-2
        xe_table = xe_features[xe_table_start:xe_table_end].split("\n")
        xe_feat_table = Table.read(xe_table, format="ascii.fixed_width_no_header", 
                                   names=("Pixel", "Fit", "User", "Fwidth", "Wt"), 
                                   col_starts=(0, 15, 26, 37, 43), col_ends=(14, 25, 36, 40, 44))
        # Now get the full feature table
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, featfile)) as ffeat:
            full_feat = ffeat.read()
        feat_entry = full_feat[full_feat.rindex("Features identified"):]
        full_feat = Table.read(feat_entry, format="ascii.fixed_width", header_start=1, data_end=-1, guess=False, 
                               col_starts=(2, 11, 22, 33), col_ends=(10, 21, 32, 43))
        # Now break it up by element.
        ne_feats = join(full_feat, ne_feat_table[["User"]])
        xe_feats = join(full_feat, xe_feat_table[["User"]])
        ar_feats = join(full_feat, ar_feat_table[["User"]])                          
        
        # Now plot the residuals
        plt.plot(ne_feats["User"], -ne_feats["Residual"], color=nc, marker="o", ls="", label="Neon")
        plt.plot(xe_feats["User"], -xe_feats["Residual"], color=xc, marker="d", ls="", label="Xenon")
        plt.plot(ar_feats["User"], -ar_feats["Residual"], color=ac, marker="s", ls="", label="Argon")
        plt.xlabel("Wavelength (A)")
        plt.ylabel("Residual (User - Fit)")
        plt.title("Full Fit Residual for {0}".format(os.path.basename(fullimg)))
        plt.legend(loc="upper left")
        plt.savefig(os.path.join(calib_residual_path, os.path.splitext(os.path.basename(featfile))[0]+".png"))
        plt.close()

In [34]:
# Check to see if the wavelength calibration along all the lamps remains steady through all nights.
sample = "night12/night12.123.fit"
for spec  in arctypes:
    night_calibration_file = "Calibration_{0}_Nights.txt".format(spec.capitalize())
    calibration_output_file = "Calibration_{0}_Extracted.txt".format(spec.capitalize())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, night_calibration_file), 'w') as repeatfile,\
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibration_output_file), 'w') as extractfile:
            for n in obsnights:
                arcfile = os.path.join("Calibrations", "Night{0}_{1}.fit".format(n, spec.capitalize()))
                outputfile = os.path.join("Calibrations", "wavelength_stability", 
                                          "Night{0}_{1}.ms.fits".format(n, spec.capitalize())) 
                repeatfile.write(arcfile+"\n")
                extractfile.write(outputfile+"\n")
                iraf.apall(arcfile, out=outputfile, ref=sample, recen=False, trace=False, back="none", intera=False)

    # Subtract the continuum from the lines.
    subtracted_calib_filelist = "Calibration_{0}_Subtracted.txt".format(spec.capitalize())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibration_output_file), 'r') as oldfile,\
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, subtracted_calib_filelist), 'w') as newfile:
            for oldname in oldfile:
                # turn Night12_Ar.ms.fits to Night12_Ar_Subtracted.ms.fits
                newname = oldname.replace(".ms.fits", "_Subtracted.ms.fits")
                newfile.write(newname)
                try:
                    os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                    print "Removed " + newname
                except OSError:
                    pass
    iraf.continuum.func = "chebyshev"
    iraf.continuum.order = 15
    iraf.continuum.high_rej = 3
    iraf.continuum.low_rej = 0
    # Continuum isn't happy with empty files. So ignore this if it's empty.
    try:
        iraf.continuum("@"+calibration_output_file, "@"+subtracted_calib_filelist, intera="no")
    except iraf.IrafError:
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibration_output_file), 'r') as infile:
            contents = infile.readlines()
            if not contents:
                pass
            else:
                raise

    # Now reidentify the lines
    iraf.reidentify(os.path.join("calib_test", "{0}spec".format(spec)), "@"+subtracted_calib_filelist, intera="no")
        
# Combine the line identifications and refit the wavelength solution.
fullspec_filelist = "Calibration_Fullspec.txt"
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, "Calibration_Ar_Subtracted.txt"), "r") as oldfile, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist), "w") as newfile:
        for oldname in oldfile:
            # turn night12.ar212.ms.fits to night12.full212.ms.fits
            newname = oldname.replace("Ar_Subtracted", "Full")
            shutil.copy(os.path.join(IMAGE_PATH, CALIB_FOLDER, oldname[:-1]), 
                        os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
            newfile.write(newname)
# Read in all of the features

with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist), "r") as fullspecs:
    for fullimg in fullspecs:
        fulldb, ext = os.path.splitext(os.path.join("database", "id"+fullimg))
        full_spec_table = []
        for spec in ["ne", "ar", "xe"]:
            specimg = fullimg.replace("Full", "{0}_Subtracted".format(spec.capitalize()))
            specdb, ext = os.path.splitext(os.path.join("database", "id"+specimg))
            # Read in the entry.
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, specdb)) as specdata:
                spec_fullfile = specdata.read()
            spec_features = spec_fullfile[spec_fullfile.rindex("begin"):]
            spec_length_line_start = spec_features.index("features")
            spec_length_line_end = spec_features.index("\n", spec_length_line_start)
            spec_numlines = int(spec_features[spec_length_line_start:spec_length_line_end].split("\t")[1])
            spec_table_start = spec_length_line_end+1
            # Subtract two because there is a trailing tab before "function"
            spec_table_end = spec_features.index("function")-2
            spec_table = spec_features[spec_table_start:spec_table_end].split("\n")
            full_spec_table = full_spec_table + spec_table
        # Now combine them
        full_numlines = len(full_spec_table)
        full_feat_table = Table.read(full_spec_table, format="ascii.fixed_width_no_header", 
                                     names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"), 
                                     col_starts=(0, 15, 26, 37, 43, 45), col_ends=(14, 25, 36, 40, 44, 46))
        print(full_feat_table)
        full_feat_table["Count"] = np.arange(len(full_feat_table))
        full_feat_table.sort("Pixel")
        sorted_table = [full_spec_table[i] for i in full_feat_table["Count"]]
        new_feature_table = "\n".join(sorted_table)

        # Now let's piece together the new file. First make the time comment.
        a = datetime.now()
        comment_line = "# " + a.strftime("%a %H:%M:%S %d-%b-%Y") + "\n"
        # Then make the header:
        spec_head_start = 0
        # Note that this includes the leading tab character in the header, not as part of the "feature" line.
        spec_head_end = spec_length_line_start
        spec_header = spec_features[spec_head_start:spec_head_end]
        full_header = spec_header.replace("{0}_Subtracted".format(spec.capitalize()), "Full")
        # Now the feature line will be added on.
        full_feature_line = "features\t{0:d}\n".format(full_numlines)
        # Lastly we want the footer, which doesn't actually contain any useful information, but we will include.
        footer = spec_features[spec_table_end:]
        # Now add them all together!
        fullfile = comment_line + full_header + full_feature_line + new_feature_table + footer
        
        # Write the result to a file.
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fulldb), 'a') as fulldata:
            fulldata.write(fullfile)

Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night1_Ne.ms already exists
Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night3_Ne.ms already exists
Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night4_Ne.ms already exists
Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night5_Ne.ms already exists
Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night6_Ne.ms already exists
Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night7_Ne.ms already exists
Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night8_Ne.ms already exists
Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night9_Ne.ms already exists
Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night10_Ne.ms already exists
Sep 25 21:16: EXTRACT - Output spectrum Calibrations/wavelength_stability/Night11

In [35]:
# Now read in all of the tables.
fullspec_filelist = "Calibration_Fullspec.txt"
night_tabledict = {}
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist), "r") as fullspecs:
    for fullimg in fullspecs:
        specdb, ext = os.path.splitext(os.path.join("database", "id"+fullimg))
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, specdb)) as specdata:
                spec_fullfile = specdata.read()
        print spec_fullfile
        spec_features = spec_fullfile[spec_fullfile.rindex("begin"):]
        spec_length_line_start = spec_features.index("features")
        spec_length_line_end = spec_features.index("\n", spec_length_line_start)
        spec_numlines = int(spec_features[spec_length_line_start:spec_length_line_end].split("\t")[1])
        spec_table_start = spec_length_line_end+1
        # Subtract two because there is a trailing tab before "function"
        spec_table_end = spec_features.index("function")-2
        spec_table = spec_features[spec_table_start:spec_table_end].split("\n")
        feat_table = Table.read(spec_table, format="ascii.fixed_width_no_header", 
                                names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"), 
                                col_starts=(0, 15, 26, 37, 43, 45), col_ends=(14, 25, 36, 40, 44, 46))
        imgbase = os.path.basename(fullimg)
        night = int(imgbase[5:imgbase.index("_")])
        night_tabledict[night] = feat_table

# Mon 20:36:54 25-Sep-2017
begin	identify Calibrations/wavelength_stability/Night1_Full.ms - Ap 1
	id	Calibrations/wavelength_stability/Night1_Full.ms
	task	identify
	image	Calibrations/wavelength_stability/Night1_Full.ms - Ap 1
	aperture	1
	aplow	137.75
	aphigh	161.75
	units	Angstroms
	features	81
	          3.94 6059.33128   6059.372   5.0 1 1 Ar I
	         20.57 6043.24471  6043.2233   5.0 1 1 ArI
	         32.02 6032.14976   6032.127   5.0 1 1 Ar I
	         34.29  6029.9841  6029.9969   5.0 1 1 
	         77.42 5987.91633  5987.9074   5.0 1 1 Ne I
	         90.09 5975.52588   5975.534   5.0 1 1 Ne I
	        100.30 5965.53713   5965.471   5.0 1 1 Ne I
	        121.43 5944.82414  5944.8342   5.0 1 1 NeI
	        137.79 5928.78445   5928.813   5.0 1 1 ArI
	        154.71  5912.1336   5912.085   5.0 1 1 Ar I
	        160.58 5906.34163  5906.4294   5.0 1 1 Ne I
	        164.50 5902.47841  5902.4623   5.0 1 1 Ne I
	        172.05 5894.96364    5894.99   5.0 1 1 Xe I
	        178.57 58

In [66]:
pixelvals = [night_tabledict[n]["Pixel"] for n in obsnights]
valarray = np.array(pixelvals)
print(valarray-np.mean(valarray, axis=0))
for i, n in enumerate(obsnights):
    diff = valarray - np.mean(valarray, axis=0)
    plt.plot(night_tabledict[n]["User"], diff[i,:]-i, 'k-', marker=".")
plt.title("Zenith Wavelength Calibration")
plt.xlabel("Feature Wavelength")
plt.ylabel("Residual from Mean Solution - Night #")

[[-0.5        -0.49461538 -0.43538462 ...,  1.65769231  1.65615385
   1.75153846]
 [ 0.         -0.02461538 -0.00538462 ..., -0.24230769 -0.25384615
  -0.22846154]
 [-0.02       -0.06461538 -0.02538462 ..., -0.23230769 -0.25384615
  -0.24846154]
 ..., 
 [ 0.13        0.19538462  0.12461538 ...,  0.01769231 -0.00384615
  -0.01846154]
 [-0.22       -0.35461538 -0.21538462 ..., -0.45230769 -0.43384615
  -0.42846154]
 [-0.3        -0.41461538 -0.29538462 ..., -0.51230769 -0.49384615
  -0.50846154]]



# Spectral mismatch variation

In [6]:
# Read in the Standard information
standard_info = Table.read(os.path.join(BINARY_PATH, "Don_May_MDM_run", "Standard_SIMBAD.txt"), 
                            format="ascii.commented_header", header_start=0, data_start=4, data_end=-1, delimiter="|", 
                           fill_values=[("~", 0), ("", 0)], guess=False)
rv_lookup = dict(zip(standard_info["typed ident"], standard_info["radvel"]))
coord_lookup = dict(zip(standard_info["typed ident"], SkyCoord(standard_info["coord1 (ICRS,J2000/2000)"], 
                                                               unit=(u.hourangle, u.deg))))

In [8]:
rv_lookup["HD185295"]

-18.574999999999999

In [7]:
file_lookup = collections.defaultdict(list)
for n in obsnights:
    for obj in obj_types:
        target_file = linearized_target_template.format(n, obj)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_file)) as targets:
            for targ in targets:
                objname = iraf.hedit(targ[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip()
                file_lookup[objname].append(targ[:-1])

In [49]:
# Check that all of the exposures are actually pointing at the objects they claim to be.
for n in obsnights:
    standards = calibrated_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standards)) as stand_file:
        for stand in stand_file:
            objlabel = iraf.hedit(stand[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip()
            ra = iraf.hedit(stand[:-1], "RA", ".", Stdout=1)[0].split("=")[1].strip()
            dec = iraf.hedit(stand[:-1], "DEC", ".", Stdout=1)[0].split("=")[1].strip()
            filecoord = SkyCoord(ra, dec, unit=(u.hourangle, u.deg))
            standard_coord = coord_lookup[objlabel]
            offset = filecoord.separation(standard_coord)
            if offset > 2*u.arcmin:
                print "{0} offset is: {1}'".format(stand[:-1], offset.to(u.arcmin))

In [23]:
# Insert the known velocity in all of the targets.
for n in obsnights:
    standards_list = linearized_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standards_list)) as standimages:
        for stand in standimages:
            objlabel = iraf.hedit(stand[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip()
            try:
                rv = rv_lookup[objlabel]
            except KeyError:
                print "Could not find entry for {0} in file {1}".format(objlabel, stand[:-1])
            iraf.hedit(stand[:-1], "VHELIO", str(rv), add="yes", verify="no")
            print(stand[:-1], rv)

add night1/night1.l072.ms.fits,VHELIO = -29.39
night1/night1.l072.ms.fits updated
('night1/night1.l072.ms.fits', -29.390000000000001)
add night1/night1.ld01.ms.fits,VHELIO = -7.73
night1/night1.ld01.ms.fits updated
('night1/night1.ld01.ms.fits', -7.7300000000000004)
add night1/night1.l077.ms.fits,VHELIO = -4.79
night1/night1.l077.ms.fits updated
('night1/night1.l077.ms.fits', -4.79)
add night1/night1.ld02.ms.fits,VHELIO = -45.55
night1/night1.ld02.ms.fits updated
('night1/night1.ld02.ms.fits', -45.549999999999997)
add night1/night1.l115.ms.fits,VHELIO = -32.83
night1/night1.l115.ms.fits updated
('night1/night1.l115.ms.fits', -32.829999999999998)
add night1/night1.l118.ms.fits,VHELIO = 5.4
night1/night1.l118.ms.fits updated
('night1/night1.l118.ms.fits', 5.4000000000000004)
add night1/night1.l119.ms.fits,VHELIO = -17.95
night1/night1.l119.ms.fits updated
('night1/night1.l119.ms.fits', -17.949999999999999)
add night1/night1.ld03.ms.fits,VHELIO = 19.98
night1/night1.ld03.ms.fits updated
(

('night4/night4.l119.ms.fits', -7.7300000000000004)
add night4/night4.l120.ms.fits,VHELIO = -73.03
night4/night4.l120.ms.fits updated
('night4/night4.l120.ms.fits', -73.030000000000001)
add night4/night4.l123.ms.fits,VHELIO = -59.51
night4/night4.l123.ms.fits updated
('night4/night4.l123.ms.fits', -59.509999999999998)
add night4/night4.l124.ms.fits,VHELIO = -44.55
night4/night4.l124.ms.fits updated
('night4/night4.l124.ms.fits', -44.549999999999997)
add night4/night4.l127.ms.fits,VHELIO = 19.98
night4/night4.l127.ms.fits updated
('night4/night4.l127.ms.fits', 19.98)
add night4/night4.l128.ms.fits,VHELIO = -94.06
night4/night4.l128.ms.fits updated
('night4/night4.l128.ms.fits', -94.060000000000002)
add night4/night4.l131.ms.fits,VHELIO = -4.79
night4/night4.l131.ms.fits updated
('night4/night4.l131.ms.fits', -4.79)
add night4/night4.l132.ms.fits,VHELIO = -66.1
night4/night4.l132.ms.fits updated
('night4/night4.l132.ms.fits', -66.099999999999994)
add night4/night4.l135.ms.fits,VHELIO = 9

('night5/night5.l206.ms.fits', -45.549999999999997)
add night5/night5.l207.ms.fits,VHELIO = -94.06
night5/night5.l207.ms.fits updated
('night5/night5.l207.ms.fits', -94.060000000000002)
add night5/night5.l210.ms.fits,VHELIO = -10.22
night5/night5.l210.ms.fits updated
('night5/night5.l210.ms.fits', -10.220000000000001)
add night5/night5.l211.ms.fits,VHELIO = -24.5
night5/night5.l211.ms.fits updated
('night5/night5.l211.ms.fits', -24.5)
add night5/night5.l214.ms.fits,VHELIO = -42.93
night5/night5.l214.ms.fits updated
('night5/night5.l214.ms.fits', -42.93)
add night5/night5.l215.ms.fits,VHELIO = -42.42
night5/night5.l215.ms.fits updated
('night5/night5.l215.ms.fits', -42.420000000000002)
add night5/night5.l218.ms.fits,VHELIO = 19.98
night5/night5.l218.ms.fits updated
('night5/night5.l218.ms.fits', 19.98)
add night5/night5.l219.ms.fits,VHELIO = -66.1
night5/night5.l219.ms.fits updated
('night5/night5.l219.ms.fits', -66.099999999999994)
add night5/night5.l222.ms.fits,VHELIO = -32.83
night5/

('night6/night6.l262.ms.fits', -41.079999999999998)
add night6/night6.l265.ms.fits,VHELIO = -121.19
night6/night6.l265.ms.fits updated
('night6/night6.l265.ms.fits', -121.19)
add night6/night6.l266.ms.fits,VHELIO = -46.66
night6/night6.l266.ms.fits updated
('night6/night6.l266.ms.fits', -46.659999999999997)
add night6/night6.l269.ms.fits,VHELIO = -6.
night6/night6.l269.ms.fits updated
('night6/night6.l269.ms.fits', -6.0)
add night6/night6.l270.ms.fits,VHELIO = -139.69
night6/night6.l270.ms.fits updated
('night6/night6.l270.ms.fits', -139.69)
add night6/night6.l273.ms.fits,VHELIO = -20.45
night6/night6.l273.ms.fits updated
('night6/night6.l273.ms.fits', -20.449999999999999)
add night6/night6.l274.ms.fits,VHELIO = -0.45
night6/night6.l274.ms.fits updated
('night6/night6.l274.ms.fits', -0.45000000000000001)
add night6/night6.l277.ms.fits,VHELIO = 8.25
night6/night6.l277.ms.fits updated
('night6/night6.l277.ms.fits', 8.25)
add night6/night6.l278.ms.fits,VHELIO = -81.18
night6/night6.l278.m

add night9/night9.l210.ms.fits,VHELIO = -46.66
night9/night9.l210.ms.fits updated
('night9/night9.l210.ms.fits', -46.659999999999997)
add night9/night9.l213.ms.fits,VHELIO = -6.
night9/night9.l213.ms.fits updated
('night9/night9.l213.ms.fits', -6.0)
add night9/night9.l214.ms.fits,VHELIO = -139.69
night9/night9.l214.ms.fits updated
('night9/night9.l214.ms.fits', -139.69)
add night9/night9.l217.ms.fits,VHELIO = -20.45
night9/night9.l217.ms.fits updated
('night9/night9.l217.ms.fits', -20.449999999999999)
add night9/night9.l218.ms.fits,VHELIO = -0.45
night9/night9.l218.ms.fits updated
('night9/night9.l218.ms.fits', -0.45000000000000001)
add night9/night9.l221.ms.fits,VHELIO = 8.25
night9/night9.l221.ms.fits updated
('night9/night9.l221.ms.fits', 8.25)
add night9/night9.l222.ms.fits,VHELIO = -81.18
night9/night9.l222.ms.fits updated
('night9/night9.l222.ms.fits', -81.180000000000007)
add night10/night10.l071.ms.fits,VHELIO = -25.83
night10/night10.l071.ms.fits updated
('night10/night10.l071

add night11/night11.l089.ms.fits,VHELIO = -2.36
night11/night11.l089.ms.fits updated
('night11/night11.l089.ms.fits', -2.3599999999999999)
add night11/night11.l092.ms.fits,VHELIO = -35.05
night11/night11.l092.ms.fits updated
('night11/night11.l092.ms.fits', -35.049999999999997)
add night11/night11.l093.ms.fits,VHELIO = -3.6
night11/night11.l093.ms.fits updated
('night11/night11.l093.ms.fits', -3.6000000000000001)
add night11/night11.l096.ms.fits,VHELIO = -39.32
night11/night11.l096.ms.fits updated
('night11/night11.l096.ms.fits', -39.32)
add night11/night11.l097.ms.fits,VHELIO = -6.82
night11/night11.l097.ms.fits updated
('night11/night11.l097.ms.fits', -6.8200000000000003)
add night11/night11.l100.ms.fits,VHELIO = 3.4
night11/night11.l100.ms.fits updated
('night11/night11.l100.ms.fits', 3.3999999999999999)
add night11/night11.l101.ms.fits,VHELIO = -30.
night11/night11.l101.ms.fits updated
('night11/night11.l101.ms.fits', -30.0)
add night11/night11.l104.ms.fits,VHELIO = -28.02
night11/

('night12/night12.l119.ms.fits', -14.69)
add night12/night12.l120.ms.fits,VHELIO = -30.62
night12/night12.l120.ms.fits updated
('night12/night12.l120.ms.fits', -30.620000000000001)
add night12/night12.l123.ms.fits,VHELIO = -42.04
night12/night12.l123.ms.fits updated
('night12/night12.l123.ms.fits', -42.039999999999999)
add night12/night12.l124.ms.fits,VHELIO = -32.87
night12/night12.l124.ms.fits updated
('night12/night12.l124.ms.fits', -32.869999999999997)
add night12/night12.l127.ms.fits,VHELIO = -15.59
night12/night12.l127.ms.fits updated
('night12/night12.l127.ms.fits', -15.59)
add night12/night12.l128.ms.fits,VHELIO = -73.03
night12/night12.l128.ms.fits updated
('night12/night12.l128.ms.fits', -73.030000000000001)
add night12/night12.l131.ms.fits,VHELIO = -29.39
night12/night12.l131.ms.fits updated
('night12/night12.l131.ms.fits', -29.390000000000001)
add night12/night12.l132.ms.fits,VHELIO = 11.38
night12/night12.l132.ms.fits updated
('night12/night12.l132.ms.fits', 11.38000000000

In [10]:
standard_name_list_lookup = {}
for n in obsnights:
    standfile = calibrated_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standfile)) as stands:
        standard_name_list_lookup[n] = [
            iraf.hedit(stand[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip() for stand in stands]

In [11]:
# Select the median J-K star from each night.
standard_JK = standard_info["Mag J"] - standard_info["Mag K"]
JK_lookup = dict(zip(standard_info["typed ident"], standard_JK))

In [44]:
file_lookup["HD86440"]

[]

NameError: name 'names' is not defined

NameError: name 'standards_crosscor_doc' is not defined

In [22]:
# Now apply the wavelength calibration to the images.
# First associate each object spectrum with the arc spectrum.
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_standards)) as fullspec, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_standards)) as stand, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_arcs)) as exarcs, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, argon_specs)) as arspec:
        for ref, obj, arc, ar in zip(fullspec, stand, exarcs, arspec):
            print obj[:-1]
            iraf.hedit(obj[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
            iraf.hedit(arc[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
            iraf.hedit(ar[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_standards)) as calstand, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_arcs)) as calarcs, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_argon_spec)) as calar:
    for cstan in itertools.chain(calstand, calarcs, calar):
        try:
            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, cstan[:-1]))
        except OSError:
            pass
iraf.dispcor("@"+extracted_standards, "@"+calibrated_standards, linearize=True)
iraf.dispcor("@"+extracted_arcs, "@"+calibrated_arcs, linearize=True)
iraf.dispcor("@"+argon_specs, "@"+calibrated_argon_spec, linearize=True)

night12/night12.076.ms.fits
night12/night12.076.ms.fits,REFSPEC1: night12/night12.full076.ms.fits -> night12/night12.full076.ms.fits
night12/night12.076.ms.fits updated
night12/night12.077t076.fits,REFSPEC1: night12/night12.full076.ms.fits -> night12/night12.full076.ms.fits
night12/night12.077t076.fits updated
night12/night12.ar076.ms.fits,REFSPEC1: night12/night12.full076.ms.fits -> night12/night12.full076.ms.fits
night12/night12.ar076.ms.fits updated
night12/night12.079.ms.fits
night12/night12.079.ms.fits,REFSPEC1: night12/night12.full079.ms.fits -> night12/night12.full079.ms.fits
night12/night12.079.ms.fits updated
night12/night12.078t079.fits,REFSPEC1: night12/night12.full079.ms.fits -> night12/night12.full079.ms.fits
night12/night12.078t079.fits updated
night12/night12.ar079.ms.fits,REFSPEC1: night12/night12.full079.ms.fits -> night12/night12.full079.ms.fits
night12/night12.ar079.ms.fits updated
night12/night12.080.ms.fits
night12/night12.080.ms.fits,REFSPEC1: night12/night12.full

night12/night12.112.ms.fits
night12/night12.112.ms.fits,REFSPEC1: night12/night12.full112.ms.fits -> night12/night12.full112.ms.fits
night12/night12.112.ms.fits updated
night12/night12.113t112.fits,REFSPEC1: night12/night12.full112.ms.fits -> night12/night12.full112.ms.fits
night12/night12.113t112.fits updated
night12/night12.ar112.ms.fits,REFSPEC1: night12/night12.full112.ms.fits -> night12/night12.full112.ms.fits
night12/night12.ar112.ms.fits updated
night12/night12.115.ms.fits
night12/night12.115.ms.fits,REFSPEC1: night12/night12.full115.ms.fits -> night12/night12.full115.ms.fits
night12/night12.115.ms.fits updated
night12/night12.114t115.fits,REFSPEC1: night12/night12.full115.ms.fits -> night12/night12.full115.ms.fits
night12/night12.114t115.fits updated
night12/night12.ar115.ms.fits,REFSPEC1: night12/night12.full115.ms.fits -> night12/night12.full115.ms.fits
night12/night12.ar115.ms.fits updated
night12/night12.116.ms.fits
night12/night12.116.ms.fits,REFSPEC1: night12/night12.full

night12/night12.210.ms.fits
night12/night12.210.ms.fits,REFSPEC1: night12/night12.full210.ms.fits -> night12/night12.full210.ms.fits
night12/night12.210.ms.fits updated
night12/night12.211t210.fits,REFSPEC1: night12/night12.full210.ms.fits -> night12/night12.full210.ms.fits
night12/night12.211t210.fits updated
night12/night12.ar210.ms.fits,REFSPEC1: night12/night12.full210.ms.fits -> night12/night12.full210.ms.fits
night12/night12.ar210.ms.fits updated
night12/night12.213.ms.fits
night12/night12.213.ms.fits,REFSPEC1: night12/night12.full213.ms.fits -> night12/night12.full213.ms.fits
night12/night12.213.ms.fits updated
night12/night12.212t213.fits,REFSPEC1: night12/night12.full213.ms.fits -> night12/night12.full213.ms.fits
night12/night12.212t213.fits updated
night12/night12.ar213.ms.fits,REFSPEC1: night12/night12.full213.ms.fits -> night12/night12.full213.ms.fits
night12/night12.ar213.ms.fits updated
night12/night12.214.ms.fits
night12/night12.214.ms.fits,REFSPEC1: night12/night12.full

night12/night12.099.ms.fits: REFSPEC1 = 'night12/night12.full099.ms.fits 1.'
night12/night12.c099.ms.fit: ap = 1, w1 = 4346.699, w2 = 6062.857, dw = 1.010099, nw = 1700
night12/night12.100.ms.fits: REFSPEC1 = 'night12/night12.full100.ms.fits 1.'
night12/night12.c100.ms.fit: ap = 1, w1 = 4346.648, w2 = 6062.877, dw =  1.01014, nw = 1700
night12/night12.103.ms.fits: REFSPEC1 = 'night12/night12.full103.ms.fits 1.'
night12/night12.c103.ms.fit: ap = 1, w1 = 4348.703, w2 = 6062.816, dw = 1.008895, nw = 1700
night12/night12.104.ms.fits: REFSPEC1 = 'night12/night12.full104.ms.fits 1.'
night12/night12.c104.ms.fit: ap = 1, w1 = 4347.442, w2 = 6062.802, dw =  1.00963, nw = 1700
night12/night12.107.ms.fits: REFSPEC1 = 'night12/night12.full107.ms.fits 1.'
night12/night12.c107.ms.fit: ap = 1, w1 = 4346.895, w2 = 6062.852, dw = 1.009981, nw = 1700
night12/night12.108.ms.fits: REFSPEC1 = 'night12/night12.full108.ms.fits 1.'
night12/night12.c108.ms.fit: ap = 1, w1 = 4347.546, w2 =  6062.85, dw = 1.0095

night12/night12.c094t095.ms.fits: ap = 1, w1 = 4347.328, w2 = 6062.814, dw = 1.009704, nw = 1700
night12/night12.097t096.fits: REFSPEC1 = 'night12/night12.full096.ms.fits 1.'
night12/night12.c097t096.ms.fits: ap = 1, w1 =   4347.3, w2 = 6062.816, dw = 1.009721, nw = 1700
night12/night12.098t099.fits: REFSPEC1 = 'night12/night12.full099.ms.fits 1.'
night12/night12.c098t099.ms.fits: ap = 1, w1 = 4346.699, w2 = 6062.857, dw = 1.010099, nw = 1700
night12/night12.101t100.fits: REFSPEC1 = 'night12/night12.full100.ms.fits 1.'
night12/night12.c101t100.ms.fits: ap = 1, w1 = 4346.648, w2 = 6062.877, dw =  1.01014, nw = 1700
night12/night12.102t103.fits: REFSPEC1 = 'night12/night12.full103.ms.fits 1.'
night12/night12.c102t103.ms.fits: ap = 1, w1 = 4348.703, w2 = 6062.816, dw = 1.008895, nw = 1700
night12/night12.105t104.fits: REFSPEC1 = 'night12/night12.full104.ms.fits 1.'
night12/night12.c105t104.ms.fits: ap = 1, w1 = 4347.442, w2 = 6062.802, dw =  1.00963, nw = 1700
night12/night12.106t107.fits

night12/night12.car088.fits: ap = 1, w1 = 4346.902, w2 = 6062.873, dw = 1.009989, nw = 1700
night12/night12.ar091.ms.fits: REFSPEC1 = 'night12/night12.full091.ms.fits 1.'
night12/night12.car091.fits: ap = 1, w1 = 4346.863, w2 = 6062.853, dw =     1.01, nw = 1700
night12/night12.ar092.ms.fits: REFSPEC1 = 'night12/night12.full092.ms.fits 1.'
night12/night12.car092.fits: ap = 1, w1 = 4346.607, w2 = 6062.877, dw = 1.010165, nw = 1700
night12/night12.ar095.ms.fits: REFSPEC1 = 'night12/night12.full095.ms.fits 1.'
night12/night12.car095.fits: ap = 1, w1 = 4347.328, w2 = 6062.814, dw = 1.009704, nw = 1700
night12/night12.ar096.ms.fits: REFSPEC1 = 'night12/night12.full096.ms.fits 1.'
night12/night12.car096.fits: ap = 1, w1 =   4347.3, w2 = 6062.816, dw = 1.009721, nw = 1700
night12/night12.ar099.ms.fits: REFSPEC1 = 'night12/night12.full099.ms.fits 1.'
night12/night12.car099.fits: ap = 1, w1 = 4346.699, w2 = 6062.857, dw = 1.010099, nw = 1700
night12/night12.ar100.ms.fits: REFSPEC1 = 'night12/ni

In [12]:
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_standards)) as calstand, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_arcs)) as exarcs:
        for stand, arcname in zip(calstand, exarcs):
            crosscor = os.path.splitext(arcname)[0] + ".txt"
            shift_table = Table.read(crosscor, format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                                     header_start=13, guess=False, 
                                     names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 
                                            'TDR', 'VOBS', 'VREL', 'VHELIO', 'VERR'])
            iraf.hedit(stand[:-1], "CRPIX1", "(1-{0:g})".format(shift_table["SHIFT"][-1]), verify=False)

IOError: [Errno 2] No such file or directory: 'night12/night12.077t076.ms.txt'

In [40]:
standard_photometry = Table.read(os.path.join(BINARY_PATH, "Don_May_MDM_run", "Standard_Photometry.txt"), 
                            format="ascii.commented_header", header_start=0, data_start=4, data_end=-1, delimiter="|", guess=False)
standard_info = Table.read(os.path.join(BINARY_PATH, "Don_May_MDM_run", "standard_csv.csv"))
standards_JK = standard_photometry["Mag J"] - standard_photometry["Mag K"]
jktable = Table([standard_photometry["typed ident"], standards_JK], names=("Star", "J-K"))
standard_info = join(jktable, standard_info, keys=["Star"])
standard_info.sort("J-K")

In [41]:
lookup = collections.defaultdict(list)
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_standards)) as calstand:
    for stand in calstand:
        hdulist = fits.open(os.path.join(IMAGE_PATH, CALIB_FOLDER, stand[:-1]))
        imgname = hdulist[0].header["OBJECT"]
        hdulist.close()
        lookup[imgname] = stand[:-1]
lookup_table = Table(rows=lookup.items(), names=("Star", "SpecPath"))
standard_info = join(standard_info, lookup_table, keys=["Star"], join_type="right")

In [86]:
for standname, rvel, standfile in zip(standard_info["Star"], standard_info["RV"], standard_info["SpecPath"]):
    iraf.hedit(standfile, "VHELIO", rvel, add=True, verify=False)

night12/night12.c226.ms.fit,VHELIO: -86 -> -86
night12/night12.c226.ms.fit updated
add night12/night12.c127.ms.fit,VHELIO = -16
night12/night12.c127.ms.fit updated
add night12/night12.c131.ms.fit,VHELIO = -29
night12/night12.c131.ms.fit updated
add night12/night12.c083.ms.fit,VHELIO = -19
night12/night12.c083.ms.fit updated
add night12/night12.c087.ms.fit,VHELIO = 5
night12/night12.c087.ms.fit updated
night12/night12.c213.ms.fit,VHELIO: -47 -> -47
night12/night12.c213.ms.fit updated
night12/night12.c234.ms.fit,VHELIO: -60 -> -60
night12/night12.c234.ms.fit updated
add night12/night12.c107.ms.fit,VHELIO = -12
night12/night12.c107.ms.fit updated
add night12/night12.c104.ms.fit,VHELIO = -2
night12/night12.c104.ms.fit updated
add night12/night12.c140.ms.fit,VHELIO = 36
night12/night12.c140.ms.fit updated
add night12/night12.c139.ms.fit,VHELIO = 15
night12/night12.c139.ms.fit updated
add night12/night12.c095.ms.fit,VHELIO = -39
night12/night12.c095.ms.fit updated
night12/night12.c222.ms.fit

In [16]:
template_standard = standard_info["Star"][len(standard_info)/2]
template_file = standard_info["SpecPath"][len(standard_info)/2]
# Get the index of the number in the name, which is assumed to be in the form of "night12/night12.c076.ms.fits"
template_num_index = template_file.index(".c")+2
template_num = template_file[template_num_index:template_num_index+3]

In [26]:
iraf.keywpar.ut = "TIME-OBS"
iraf.keywpar.epoch = "EQUINOX"

iraf.fxcor.pixcorr = False
iraf.function = "gaussian"

iraf.continpars.c_inter = True
iraf.continpars.order = 10
iraf.continpars.low_rej = 2
iraf.continpars.high_rej = 5
iraf.continpars.nitera = 10
iraf.continpars.grow = 1

iraf.filtpars.cutoff = 250
iraf.filtpars.cuton = 25

for objrow in standard_info:
    object_file = objrow["SpecPath"]
    object_num_index = object_file.index(".c")+2
    object_num = object_file[object_num_index:object_num_index+3]
    
    output_root = os.path.join("night{0:d}", "night{0}.{1}cc{2}").format(Calib_Night, object_num, template_num)
    print(output_root)
    iraf.fxcor(object_file, template_file, output=output_root)
    

night12/night12.226cc096
Cross-Correlating night12/night12.c226.ms.fit[1] with night12/night12.c096.ms.fit[1].
HJD=7915.9757  FWHM=304.43  Vr=-119.313  Vo=-101.115  Vh=-85.154 +/- 7.615
HJD=7915.9757  FWHM=330.17  Vr=-119.356  Vo=-101.159  Vh=-85.197 +/- 8.291
Writing current results to `night12/night12.226cc096.txt'....Done.
night12/night12.127cc096
Cross-Correlating night12/night12.c127.ms.fit[1] with night12/night12.c096.ms.fit[1].
HJD=7915.7002  FWHM=306.73  Vr=-9.204  Vo=9.001  Vh=-7.947 +/- 8.071
HJD=7915.7002  FWHM=326.53  Vr=-9.234  Vo=8.970  Vh=-7.977 +/- 8.618
Writing current results to `night12/night12.127cc096.txt'....Done.
night12/night12.131cc096
Cross-Correlating night12/night12.c131.ms.fit[1] with night12/night12.c096.ms.fit[1].
HJD=7915.7036  FWHM=306.50  Vr=-41.243  Vo=-23.041  Vh=-33.606 +/- 8.105
HJD=7915.7036  FWHM=327.67  Vr=-41.270  Vo=-23.068  Vh=-33.633 +/- 8.693
Writing current results to `night12/night12.131cc096.txt'....Done.
night12/night12.083cc096
Cross-C

HJD=7915.6642  FWHM=336.70  Vr=20.185  Vo=38.391  Vh=12.952 +/- 6.557
Writing current results to `night12/night12.099cc096.txt'....Done.


In [42]:
standard_info.sort("J-K")
cor_vels = np.zeros(len(standard_info))
for i, specfile in enumerate(standard_info["SpecPath"]):  
    # Now get the pixel shift.
    num_index = specfile.index(".c")+2
    specnum = specfile[num_index:num_index+3]
    shiftfile = os.path.join("night{0}", "night{0}.{1}cc{2}.txt").format(Calib_Night, specnum, template_num)
    shift_table = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, shiftfile), format="ascii.commented_header", 
                             fill_values=[("", "0"), ("INDEF", "0")], header_start=13, guess=False, 
                             names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                                    'VOBS', 'VREL', 'VHELIO', 'VERR'])
    cor_vels[i] = shift_table["VHELIO"][-1]

In [43]:
veldiffs = cor_vels - standard_info["RV"]
plt.plot(standard_info["J-K"], veldiffs, 'ko')
plt.plot(standard_info["J-K"][len(standard_info)/2], veldiffs[len(veldiffs)/2], 'r*', ms=10)
plt.xlabel("J-K")
plt.ylabel("Relative velocity offset")

In [52]:
standard_info["Star"][len(standard_info)/2]

'HD110044'

In [24]:
standard_info["RV"][len(standard_info)/2]

Star,J-K,Spectype,RV,V,SpecPath
str9,float64,str8,int64,float64,str27
HD110044,0.455,K1V,-7,9.0,night12/night12.c096.ms.fit


In [48]:
plt.plot(cor_vels, standard_info["RV"], 'ko')
plt.xlabel("Cross-correlated RV (km/s)")
plt.ylabel("Literature RV (km/s)")

# Cross-correlation

In [8]:
def compact_standard(filename):
    '''Compactify a filename.'''
    compact = os.path.splitext(os.path.splitext(os.path.basename(filename))[0])[0].replace(".", "").replace("night","n").replace("NIGHT", "N")
    return compact

In [9]:
# Dictionary filled with the reference standards for each night
def nth(iterable, n, default=None):
    "Returns the nth item or a default value"
    return next(itertools.islice(iterable, n, None), default)
reference_spectra = {}
for n in obsnights:
    stand_list = linearized_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, stand_list)) as night_stands:
        # Select the first object.
        try:
            reference_spectra[n] = nth(night_stands, 0)[:-1]
        # Night 7 doesn't have any observed standards, so use the standard for night 6 instead.
        except TypeError:
            reference_spectra[n] = reference_spectra[n-1]

In [12]:
# Calculate offset for reference standards.
# This starts with cross-correlating to each other to determine what the measured RVs are.

iraf.fxcor.high_rej = 5
iraf.fxcor.low_rej = 2
iraf.fxcor.pixcor = "no"
iraf.fxcor.function = "gaussian"

iraf.continpars.order = 15
iraf.continpars.nitera = 10
iraf.continpars.grow = 1
iraf.continpars.c_inter = True

iraf.keywpars.ut = "TIME-OBS"
iraf.keywpars.epoch = "EQUINOX"

iraf.filtpars.cutoff = 250
iraf.filtpars.cuton = 25

# Just do standard correlation from night to night.
for n, template in itertools.islice(reference_spectra.iteritems(), 2, 3):
    # Now begin going through standards for that night.
    template_cor_file = target_cor_template.format(n, obj_types[0], compact_standard(template).upper())
    target_list = linearized_target_template.format(n, obj_types[0])
    # Now populate template_cor_file
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_list), 'r') as oldfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file), "w") as newfile:
            for oldname in oldfile:
                newname = oldname.replace(".ms.fits", compact_standard(template))
                newfile.write(newname)
    # Now cross-correlate.
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_list)) as targets, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file)) as outputs:
            for targ, out in zip(targets, outputs):
                if targ[:-1] != template:
                    iraf.fxcor(targ[:-1], template, out=out[:-1], interact="yes")
                    print "Cross-correlating {0}".format(out[:-1])
                else:
                    print "Skipping {0}".format(out[:-1])

Skipping night4/night4.l078n4l078
Cross-Correlating night4/night4.l079.ms.fits[1] with night4/night4.l078.ms.fits[1].
HJD=7907.6464  FWHM=293.95  Vr=53.632  Vo=38.223  Vh=13.099 +/- 6.607


Killing IRAF task `fxcor'


KeyboardInterrupt: 

In [13]:
# Calculate offset for reference standards.
# Read in the measured RVs and calibrate them so that they are all on the Catalog scale.
reference_corrections = {}

for n, template in reference_spectra.iteritems():
    # Read in the resulting RVs
    measured_RVs = []
    template_cor_file = target_cor_template.format(n, obj_types[0], compact_standard(template).upper())
    target_list = linearized_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file)) as cor_measurements:
        for cor in cor_measurements:
            # Example filename: night6/night6.c104n6c104
            firstl = cor.rindex("l")
            secondl = cor[:firstl].rindex("l")
            targnum = cor[secondl+1:secondl+4]
            tempnum = cor[firstl+1:firstl+4]
            if targnum != tempnum:
                vel_table = Table.read(
                    os.path.join(IMAGE_PATH, CALIB_FOLDER, cor[:-1]+".txt"), format="ascii.commented_header", 
                    fill_values=[("", "0"), ("INDEF", "0")], header_start=13, guess=False, 
                    names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                           'VOBS', 'VREL', 'VHELIO', 'VERR'])
                measured_RVs.append(vel_table["VHELIO"][-1])
    measured_RVs = np.array(measured_RVs, dtype=np.float)
    # Now read in the catalog RVs
    catalog_RVs = np.zeros(measured_RVs.shape)
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_list)) as targets:
        for i, stand in enumerate(itertools.ifilter(lambda x: x[:-1] != template, targets)):
            catalog_RVs[i] = float(iraf.hedit(stand[:-1], "VHELIO", ".", Stdout=1)[0].split("=")[1])
    # Now fit the reference to the catalog in order to find the offset.
    try:
        linefit = np.polyfit(catalog_RVs, measured_RVs, 1, cov=False)
    # Night 7 should use the previous night's value.
    except TypeError:
        reference_corrections[n] = reference_corrections[n-1]
    else:
        reference_corrections[n] = - linefit[1]

In [64]:
# Now cross-correlate the Kepler stars against the reference standards.
iraf.keywpars.ut = "TIME-OBS"
iraf.keywpars.epoch = "EQUINOX"
iraf.fxcor.pixcorr = False
iraf.continpars.low_reject = 2.0
# Cosmic rays are a bigger deal here.
iraf.continpars.high_reject = 5.0
for n, template_standard in reference_spectra.iteritems():
    kicfiles = linearized_target_template.format(n, obj_types[1])
    velocity_files = target_cor_template.format(n, obj_types[1], compact_standard(template_standard).upper())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kicfiles), 'r') as oldfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, velocity_files), 'w') as newfile:
            for oldname in oldfile:
                newname = oldname.replace(".ms.fits", compact_standard(template_standard))
                newfile.write(newname)
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kicfiles)) as kic_targets, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, velocity_files)) as output_files:
            for kic_target, output in zip(kic_targets, output_files):
                iraf.fxcor(kic_target[:-1], template_standard, out=output[:-1], intera="no")

In [69]:
plt.close("all")

In [16]:
# Holds a 2-tuple with (times, rvs)
kepler_rvs = {}
for kicobj, kicfiles in itertools.ifilter(lambda x: x[0].startswith("KIC"), file_lookup.iteritems()):
    kictimes = np.zeros(len(kicfiles))
    kicrvs = np.zeros(len(kicfiles))
    for i, kicfile in enumerate(kicfiles):
        kicnum = kicfile[-11:-8]
        nightnum = int(kicfile[kicfile.index("t")+1:kicfile.index(os.sep)])
        # transform night12/night12.l135.ms.fits -> night12/night12.l135n12l077.txt
        rv_file = kicfile.replace(
            ".ms.fits", compact_standard(reference_spectra[nightnum]) +".txt")
        vel_table = Table.read(
            os.path.join(IMAGE_PATH, CALIB_FOLDER, rv_file), format="ascii.commented_header", 
            fill_values=[("", "0"), ("INDEF", "0")], header_start=13, guess=False, 
            names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                   'VOBS', 'VREL', 'VHELIO', 'VERR'])
        kicrvs[i] = vel_table["VHELIO"][-1]+reference_corrections[nightnum]
        kictimes[i] = vel_table["HJD"][-1]
    kepler_rvs[kicobj] = (kictimes, kicrvs)

In [28]:
# Calculate chi-squared for each RV curve
# Mean RV, RV Scatter, Chi-squared, Reduced Chi-squared.
rverr = 12.0
stat_list = []
for kicobj, (times, curve) in kepler_rvs.iteritems():
    # I want stat table to hold tuples
    # (objname, mean rv, rv scatter, chi-squared, reduced chi-squared)
    meanrv = np.mean(curve)
    rvscatter = np.std(curve)
    chi_sq = np.sum((curve-meanrv)**2)/rverr**2
    # Subtract one degree of freedom because we don't know the RVs a priori.
    reduced_chi_sq = chi_sq / (len(curve)-1)
    stat_list.append((kicobj, meanrv, rvscatter, chi_sq, reduced_chi_sq))
    stat_table = Table(rows=stat_list, names=("Name", "Mean RV", "RV Scatter", "Chi Squared", "Reduced Chi Squared"))

In [153]:
stat_table.meta["comments"] = [
    "Statistics for the Kepler radial velocity curves.",
    "",
    "The first column is the name of the Kepler object. The second is the mean RV",
    "in km/s measured from curve. The third is the scatter about the mean RV in", 
    "km/s. The fourth is the chi-squared of the RV curve assuming that the null",
    "hypothesis is RV non-variability with an uncertainty of 12 km/s. The last ",
    "column is the reduced chi-squared of the RV curve which should provide a rough ",
    "estimation of whether the object is RV variable or not."
]
statfile = "RV_stats.txt"
formats = {"Mean RV": ".1f", "RV Scatter": ".1f", "Chi Squared": ".1f", "Reduced Chi Squared": ".2f"}
stat_table.write(os.path.join(BINARY_PATH, statfile), format="ascii.fixed_width", formats=formats, overwrite=True)

In [29]:
dofs = stat_table["Chi Squared"] / stat_table["Reduced Chi Squared"]
nonvar_probs = stats.chi2.sf(stat_table["Chi Squared"], dofs)
print stat_table["Name"][nonvar_probs < 0.05]

    Name   
-----------
KIC12736892
 KIC7421325
 KIC3248885
 KIC5609753


In [96]:
periods = {"KIC1570924": 3.234, "KIC3540728": 2.130, "KIC5553362": 4.485, "KIC7294867": 1.046, "KIC8442720": 3.504, 
           "KIC9964938": 2.893, "KIC10293980": 1.067, "KIC11073910": 2.064, "KIC11819949": 2.303, "KIC12736892": 2.650, 
           "KIC8651471": 3.487, "KIC4249702": 4.712, "KIC10802309": 1.999, "KIC6780052": 3.157, "KIC3539632": 3.101, 
           "KIC4480434": 4.535, "KIC7919763": 3.784, "KIC4454890": 2.256, "KIC3248885": 4.898, "KIC5213142": 2.511, 
           "KIC9710336": 4.460, "KIC10153521": 1.743, "KIC4036736": 2.823, "KIC6844101": 2.528, "KIC11080481": 1.012, 
           "KIC3219623": 1.684, "KIC6425783": 3.911, "KIC9655045": 2.955, "KIC9653110": 3.133, "KIC7421325": 4.771, 
           "KIC9151271": 4.633, "KIC5609753": 3.250}

In [103]:
for keplerobj in itertools.islice(kepler_rvs, 0, None):
    plt.figure()
    rverr = 12
    obstimes, rvcurve = kepler_rvs[keplerobj]
    rotperiod = periods[keplerobj]
    basetime = 7904 # The base in HJD for some reason.
    times = (obstimes - basetime) % rotperiod
    meanrv = np.mean(rvcurve)
    plt.errorbar(times, rvcurve-meanrv, yerr=rverr/2, fmt='ko')
    plt.plot([0, 14], [0, 0], 'k--')
    plt.xlabel("Phased Time (HJD)")
    plt.ylabel("Heliocentric RV Offset (km/s)")
    plt.title(keplerobj)
#    plt.xlim(0, 15)
#    plt.ylim(-50, 50)
#    plt.savefig(os.path.join(BINARY_PATH, "plots", "Kepler_RVs", keplerobj+".png"), overwrite=True)

In [81]:
don_rvs = {k:kepler_rvs[k] for k in ("KIC3219623", "KIC5609753", "KIC12736892")}
don_rvs_doc = """var_rvs.pickle
------------------------------

This file contains a tuple consisting of these instructions as the first
element, and a dictionary mapping the KIC name of the interesting RV variable
targets to a 2-tuple containing the HJD times of observation and the measured RVs

To read in the contents of var_rvs.pickle, simply use::

    import cPickle as pickle
    with open("var_rvs.pickle") as rvfile:
        _, rvdict = pickle.load(rvfile)
    kic321times, kic321rvs = rvdict["KIC3219623"]
    
"""
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, "var_rvs.pickle"), "w") as rv_output:
    pickle.dump((don_rvs_doc, don_rvs), rv_output)

In [35]:
full_rvs_doc = """full_rvs.pickle
--------------

This file contains a tuple consisting of these instructions as the first 
element, and a dictionary mapping the KIC name of the targets to a 2-tuple
conaining the HJD times of the observation and the measured RVs.

To read in the contents of full_rvs.pickle, simply use::

    import pickle
    with open("full_rvs.pickle", "rb") as rvfile:
        docs, rvdict = pickle.load(rvfile)
    kic321times, kic321rvs = rvdict["KIC3219623"]
    
"""
with open(os.path.join(BINARY_PATH, "full_rvs.pickle"), 'w') as rv_output:
    pickle.dump((full_rvs_doc, kepler_rvs), rv_output)

In [34]:
# Write README documentation
with open(os.path.join(BINARY_PATH, "README.txt"), "w") as readme:
    readme.write(full_rvs_doc)

 # Periodogram

In [92]:
lowfreq = 1.0/14
highfreq = 1
samplerate = lowfreq / 5.0
freqs = np.arange(lowfreq, highfreq, samplerate)
lombfreqs = 2*np.pi*freqs
periods = 1 / freqs
for keplerobj in itertools.islice(kepler_rvs, 0, None):
    plt.figure()
    obstimes, rvcurve = kepler_rvs[keplerobj]
    basetime = 7904 # The base in HJD for some reason.
    times = obstimes - basetime #% rotperiod
    pgram = lombscargle(times, rvcurve, lombfreqs)
    plt.plot(periods, pgram/max(pgram), 'b-')
    plt.xlabel("Period (day)")
    plt.ylabel("Spectral power")
    plt.title(keplerobj + "Periodogram")
    plt.xlim(1.5, 14)
#    plt.savefig(os.path.join(BINARY_PATH, "plots", "Kepler_RVs", keplerobj+".png"), overwrite=True)

# Error Analysis

## Full analysis

In [51]:
iraf.fxcor.high_rej = 5
iraf.fxcor.low_rej = 2
iraf.filtpars.cuton = 25
iraf.filtpars.cutoff = 250
iraf.continpars.order = 15
iraf.fxcor.pixcor = "no"
iraf.keywpars.ut = "TIME-OBS"
iraf.keywpars.epoch = "EQUINOX"

# Just do standard correlation from night to night.
for n in obsnights:
    template_list = linearized_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_list)) as templates:
        for template in templates:
            # Now begin going through targets.
            template_cor_file = target_cor_template.format(n, obj_types[0], compact_standard(template).upper())
            target_list = linearized_target_template.format(n, obj_types[0])
            # Now populate template_cor_file
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_list), 'r') as oldfile, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file), "w") as newfile:
                    for oldname in oldfile:
                        newname = oldname.replace(".ms.fits", compact_standard(template))
                        newfile.write(newname)
            # Now cross-correlate.
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_list)) as targets, \
                 open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file)) as outputs:
                    for targ, out in zip(targets, outputs):
                        if targ != template:
                            iraf.fxcor(targ[:-1], template[:-1], out=out[:-1], interact="no")
                            print "Cross-correlating {0}".format(out[:-1])
                        else:
                            print "Skipping {0}".format(out[:-1])

Skipping night1/night1.l072n1l072
Cross-correlating night1/night1.ld01n1l072
Cross-correlating night1/night1.l077n1l072
Cross-correlating night1/night1.ld02n1l072
Cross-correlating night1/night1.l115n1l072
Cross-correlating night1/night1.l118n1l072
Cross-correlating night1/night1.l119n1l072
Cross-correlating night1/night1.ld03n1l072
Cross-correlating night1/night1.l124n1l072
Cross-correlating night1/night1.ld04n1l072
Cross-correlating night1/night1.ld05n1l072
Cross-correlating night1/night1.l072n1ld01
Skipping night1/night1.ld01n1ld01
Cross-correlating night1/night1.l077n1ld01
Cross-correlating night1/night1.ld02n1ld01
Cross-correlating night1/night1.l115n1ld01
Cross-correlating night1/night1.l118n1ld01
Cross-correlating night1/night1.l119n1ld01
Cross-correlating night1/night1.ld03n1ld01
Cross-correlating night1/night1.l124n1ld01
Cross-correlating night1/night1.ld04n1ld01
Cross-correlating night1/night1.ld05n1ld01
Cross-correlating night1/night1.l072n1l077
Cross-correlating night1/nigh

Cross-correlating night3/night3.l106n3l086
Cross-correlating night3/night3.l107n3l086
Cross-correlating night3/night3.l111n3l086
Cross-correlating night3/night3.l112n3l086
Cross-correlating night3/night3.l115n3l086
Cross-correlating night3/night3.ld02n3l086
Cross-correlating night3/night3.l120n3l086
Cross-correlating night3/night3.l121n3l086
Cross-correlating night3/night3.l124n3l086
Cross-correlating night3/night3.l125n3l086
Cross-correlating night3/night3.l176n3l086
Cross-correlating night3/night3.l177n3l086
Cross-correlating night3/night3.l179n3l086
Cross-correlating night3/night3.l182n3l086
Cross-correlating night3/night3.l183n3l086
Cross-correlating night3/night3.l186n3l086
Cross-correlating night3/night3.l187n3l086
Cross-correlating night3/night3.l190n3l086
Cross-correlating night3/night3.l191n3l086
Cross-correlating night3/night3.l194n3l086
Cross-correlating night3/night3.l197n3l086
Cross-correlating night3/night3.l081n3l087
Cross-correlating night3/night3.l083n3l087
Cross-corre

Cross-correlating night3/night3.l124n3l100
Cross-correlating night3/night3.l125n3l100
Cross-correlating night3/night3.l176n3l100
Cross-correlating night3/night3.l177n3l100
Cross-correlating night3/night3.l179n3l100
Cross-correlating night3/night3.l182n3l100
Cross-correlating night3/night3.l183n3l100
Cross-correlating night3/night3.l186n3l100
Cross-correlating night3/night3.l187n3l100
Cross-correlating night3/night3.l190n3l100
Cross-correlating night3/night3.l191n3l100
Cross-correlating night3/night3.l194n3l100
Cross-correlating night3/night3.l197n3l100
Cross-correlating night3/night3.l081n3l102
Cross-correlating night3/night3.l083n3l102
Cross-correlating night3/night3.l086n3l102
Cross-correlating night3/night3.l087n3l102
Cross-correlating night3/night3.l090n3l102
Cross-correlating night3/night3.ld01n3l102
Cross-correlating night3/night3.l095n3l102
Cross-correlating night3/night3.l096n3l102
Cross-correlating night3/night3.l100n3l102
Skipping night3/night3.l102n3l102
Cross-correlating ni

Cross-correlating night3/night3.l182n3l112
Cross-correlating night3/night3.l183n3l112
Cross-correlating night3/night3.l186n3l112
Cross-correlating night3/night3.l187n3l112
Cross-correlating night3/night3.l190n3l112
Cross-correlating night3/night3.l191n3l112
Cross-correlating night3/night3.l194n3l112
Cross-correlating night3/night3.l197n3l112
Cross-correlating night3/night3.l081n3l115
Cross-correlating night3/night3.l083n3l115
Cross-correlating night3/night3.l086n3l115
Cross-correlating night3/night3.l087n3l115
Cross-correlating night3/night3.l090n3l115
Cross-correlating night3/night3.ld01n3l115
Cross-correlating night3/night3.l095n3l115
Cross-correlating night3/night3.l096n3l115
Cross-correlating night3/night3.l100n3l115
Cross-correlating night3/night3.l102n3l115
Cross-correlating night3/night3.l103n3l115
Cross-correlating night3/night3.l106n3l115
Cross-correlating night3/night3.l107n3l115
Cross-correlating night3/night3.l111n3l115
Cross-correlating night3/night3.l112n3l115
Skipping ni

Cross-correlating night3/night3.l190n3l125
Cross-correlating night3/night3.l191n3l125
Cross-correlating night3/night3.l194n3l125
Cross-correlating night3/night3.l197n3l125
Cross-correlating night3/night3.l081n3l176
Cross-correlating night3/night3.l083n3l176
Cross-correlating night3/night3.l086n3l176
Cross-correlating night3/night3.l087n3l176
Cross-correlating night3/night3.l090n3l176
Cross-correlating night3/night3.ld01n3l176
Cross-correlating night3/night3.l095n3l176
Cross-correlating night3/night3.l096n3l176
Cross-correlating night3/night3.l100n3l176
Cross-correlating night3/night3.l102n3l176
Cross-correlating night3/night3.l103n3l176
Cross-correlating night3/night3.l106n3l176
Cross-correlating night3/night3.l107n3l176
Cross-correlating night3/night3.l111n3l176
Cross-correlating night3/night3.l112n3l176
Cross-correlating night3/night3.l115n3l176
Cross-correlating night3/night3.ld02n3l176
Cross-correlating night3/night3.l120n3l176
Cross-correlating night3/night3.l121n3l176
Cross-corre

...

Cross-correlating night14/night14.l079n14l119
Cross-correlating night14/night14.l082n14l119
Cross-correlating night14/night14.l083n14l119
Cross-correlating night14/night14.l086n14l119
Cross-correlating night14/night14.l087n14l119
Cross-correlating night14/night14.l090n14l119
Cross-correlating night14/night14.l091n14l119
Cross-correlating night14/night14.l094n14l119
Cross-correlating night14/night14.l095n14l119
Cross-correlating night14/night14.l098n14l119
Cross-correlating night14/night14.l099n14l119
Cross-correlating night14/night14.l102n14l119
Cross-correlating night14/night14.l103n14l119
Cross-correlating night14/night14.l106n14l119
Cross-correlating night14/night14.l107n14l119
Cross-correlating night14/night14.l110n14l119
Cross-correlating night14/night14.l111n14l119
Cross-correlating night14/night14.l114n14l119
Cross-correlating night14/night14.l115n14l119
Cross-correlating night14/night14.l118n14l119
Skipping night14/night14.l119n14l119
Cross-correlating night14/night14.l122n14l1

Cross-correlating night14/night14.l098n14l127
Cross-correlating night14/night14.l099n14l127
Cross-correlating night14/night14.l102n14l127
Cross-correlating night14/night14.l103n14l127
Cross-correlating night14/night14.l106n14l127
Cross-correlating night14/night14.l107n14l127
Cross-correlating night14/night14.l110n14l127
Cross-correlating night14/night14.l111n14l127
Cross-correlating night14/night14.l114n14l127
Cross-correlating night14/night14.l115n14l127
Cross-correlating night14/night14.l118n14l127
Cross-correlating night14/night14.l119n14l127
Cross-correlating night14/night14.l122n14l127
Cross-correlating night14/night14.l123n14l127
Cross-correlating night14/night14.l126n14l127
Skipping night14/night14.l127n14l127
Cross-correlating night14/night14.l130n14l127
Cross-correlating night14/night14.l131n14l127
Cross-correlating night14/night14.l199n14l127
Cross-correlating night14/night14.l200n14l127
Cross-correlating night14/night14.l203n14l127
Cross-correlating night14/night14.l204n14l1

Cross-correlating night14/night14.l115n14l200
Cross-correlating night14/night14.l118n14l200
Cross-correlating night14/night14.l119n14l200
Cross-correlating night14/night14.l122n14l200
Cross-correlating night14/night14.l123n14l200
Cross-correlating night14/night14.l126n14l200
Cross-correlating night14/night14.l127n14l200
Cross-correlating night14/night14.l130n14l200
Cross-correlating night14/night14.l131n14l200
Cross-correlating night14/night14.l199n14l200
Skipping night14/night14.l200n14l200
Cross-correlating night14/night14.l203n14l200
Cross-correlating night14/night14.l204n14l200
Cross-correlating night14/night14.l207n14l200
Cross-correlating night14/night14.l208n14l200
Cross-correlating night14/night14.l211n14l200
Cross-correlating night14/night14.l212n14l200
Cross-correlating night14/night14.l215n14l200
Cross-correlating night14/night14.l216n14l200
Cross-correlating night14/night14.l219n14l200
Cross-correlating night14/night14.l220n14l200
Cross-correlating night14/night14.l223n14l2

Cross-correlating night14/night14.l211n14l208
Cross-correlating night14/night14.l212n14l208
Cross-correlating night14/night14.l215n14l208
Cross-correlating night14/night14.l216n14l208
Cross-correlating night14/night14.l219n14l208
Cross-correlating night14/night14.l220n14l208
Cross-correlating night14/night14.l223n14l208
Cross-correlating night14/night14.l224n14l208
Cross-correlating night14/night14.l075n14l211
Cross-correlating night14/night14.l078n14l211
Cross-correlating night14/night14.l079n14l211
Cross-correlating night14/night14.l082n14l211
Cross-correlating night14/night14.l083n14l211
Cross-correlating night14/night14.l086n14l211
Cross-correlating night14/night14.l087n14l211
Cross-correlating night14/night14.l090n14l211
Cross-correlating night14/night14.l091n14l211
Cross-correlating night14/night14.l094n14l211
Cross-correlating night14/night14.l095n14l211
Cross-correlating night14/night14.l098n14l211
Cross-correlating night14/night14.l099n14l211
Cross-correlating night14/night14.

Cross-correlating night14/night14.l075n14l219
Cross-correlating night14/night14.l078n14l219
Cross-correlating night14/night14.l079n14l219
Cross-correlating night14/night14.l082n14l219
Cross-correlating night14/night14.l083n14l219
Cross-correlating night14/night14.l086n14l219
Cross-correlating night14/night14.l087n14l219
Cross-correlating night14/night14.l090n14l219
Cross-correlating night14/night14.l091n14l219
Cross-correlating night14/night14.l094n14l219
Cross-correlating night14/night14.l095n14l219
Cross-correlating night14/night14.l098n14l219
Cross-correlating night14/night14.l099n14l219
Cross-correlating night14/night14.l102n14l219
Cross-correlating night14/night14.l103n14l219
Cross-correlating night14/night14.l106n14l219
Cross-correlating night14/night14.l107n14l219
Cross-correlating night14/night14.l110n14l219
Cross-correlating night14/night14.l111n14l219
Cross-correlating night14/night14.l114n14l219
Cross-correlating night14/night14.l115n14l219
Cross-correlating night14/night14.

In [19]:
full_data = {}
obstimes = collections.defaultdict(list)
for n in obsnights:
    template_list = linearized_target_template.format(n, obj_types[0])
    fullvalues = []
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_list)) as templates:
        for i, template in enumerate(templates):
            tempvalues = []
            template_cor_file = target_cor_template.format(n, obj_types[0], compact_standard(template).upper())
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file)) as cor_measurements:
                for j, cor in enumerate(cor_measurements):
                    if i != j:
                        vel_table = Table.read(
                            os.path.join(IMAGE_PATH, CALIB_FOLDER, cor[:-1]+".txt"), format="ascii.commented_header", 
                            fill_values=[("", "0"), ("INDEF", "0")], header_start=13, guess=False, 
                            names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                                   'VOBS', 'VREL', 'VHELIO', 'VERR'])
                        rv = vel_table["VHELIO"][-1]
                    else:
                        rv = float(iraf.hedit(template[:-1], "VHELIO", ".", Stdout=1)[0].split("=")[1])
                    tempvalues.append(rv)
            fullvalues.append(tempvalues)
            obstimes[n].append(float(iraf.hedit(template[:-1], "JD", ".", Stdout=1)[0].split("=")[1]))
    full_array = np.ma.array(fullvalues, mask=np.identity(len(fullvalues)))
    full_data[n] = full_array
    obstimes[n] = np.array(obstimes[n])
# Templates are rows ([0,:]). Targets are columns ([:,0]).
# The coordinate [i, j] is the RV of the target j using the template i.

In [20]:
# In this case, don't fit to true values. Fit to each other.
coefficients = {}
for n, rvsq in full_data.iteritems():
    # Since we set the diagonals to the true RV, but then masked them. They should be recoverable via the "data" attribute.
    base_rvs = np.atleast_2d(rvsq)[0,:]
    # This array holds: slope, intercept, slopeerr, intercepterr
    fitresults = np.zeros((rvsq.shape[0], 4))
    for i in xrange(rvsq.shape[0]):
        rvslice = rvsq[i,:]
        fullmask = np.logical_or(base_rvs.mask, rvslice.mask)
        removed_base_rvs = base_rvs[~fullmask]
        fitrvs = rvslice[~fullmask]
        linfit, cov = np.ma.polyfit(removed_base_rvs, fitrvs, 1, cov=True)
        fitresults[i, 0:2] = linfit
        fitresults[i, 2:4] = np.sqrt(np.diag(cov))
        plt.plot(removed_base_rvs, fitrvs, '.')
    coefficients[n] = fitresults

In [21]:
corrected_rvs = {}
for n, coeff, data in ((k, coefficients[k], v) for k, v in full_data.iteritems()):
    corrected_rvs[n] = data - coeff[:,1:2]

In [79]:
# Write an intra-night fit table.
write_tables = []
for n in obsnights:
    coeffs = coefficients[n]
    intrascat = np.ma.std(corrected_rvs[n], axis=0)
    
    coefftable = Table(coeffs, names=["Slope", "Intercept", "Slope Err", "Intercept Err"])
    coefftable["Scatter"] = intrascat
    coefftable["Night"] = n * np.ones(intrascat.shape, dtype=np.int)
    coefftable["Object"] = np.array(standard_name_list_lookup[n], dtype="U10")
    
    coefftable = coefftable["Night", "Object", "Slope", "Intercept", "Slope Err", "Intercept Err", "Scatter"]
    
    write_tables.append(coefftable)
    
fulltable = vstack(write_tables)
fulltable.meta["comments"] = ["Fit parameters between templates for a given night.",
"",
"The first four columns of the file are the fit values between each standard and",
'the "reference" standard. The last column is the scatter remaining between the', 
'standards after the zero-point difference between them and the "reference"',
'standard is corrected for.',
'',
'For this particular case, the "reference" standard is the first standard',
'observed during the night. For this reason, the parameters for the first',
'standard of the night should be unity slope, zero intercept.']
fulltable.write(os.path.join(IMAGE_PATH, CALIB_FOLDER, "RV_Intranight_Params.txt"), format="ascii.fixed_width", 
                include_names=["Night", "Object", "Slope", "Intercept", "Slope Err", "Intercept Err", "Scatter"])
    

NameError: name 'standard_name_list_lookup' is not defined

In [63]:
print reference_corrections
print reference_offset

{1: 1.8145939489874188, 3: 0.36985361985921705, 4: -15.825994648433545, 5: -21.375979811186461, 6: -8.4606618385181864, 7: -8.4606618385181864, 8: 22.912317095143656, 9: 6.6963417738040452, 10: 2.8101150036664317, 11: 5.1179584508674578, 12: 7.5705575766178628, 13: -3.1295292563268005, 14: 29.611031108585003}
{1: 1.8847402871497418, 3: 0.43487053499687628, 4: -15.561937953945359, 5: -21.055959523725953, 6: -8.2059181200890823, 8: 22.116478259400996, 9: 6.5514976204535422, 10: 2.7222546742858804, 11: 5.0015251263323774, 12: 7.3824174313756465, 13: -3.0855382968798422, 14: 28.499486926715758}


In [22]:
template

'night1/night1.l072.ms.fits\n'

In [22]:
reference_offset = {}
displacements = {}
nightfit_results = np.ma.zeros((len(obsnights), 4))
for i, n in enumerate(obsnights):
    plt.figure()
    rvsq = full_data[n]
    fitcoeff = coefficients[n]
    # Instead of plotting multiple times, just plot all of the data points at once.
    cor_rvs = np.ma.mean(corrected_rvs[n], axis=0)
    true_rvs = np.diag(rvsq.data)
    dupmask = ~cor_rvs.mask
    measured_rvs = cor_rvs[dupmask]
    catalog_rvs = true_rvs[dupmask]
    #plt.plot(true_rvs[1,:], corrected_rvs[1,:], 'k.')
    #plt.plot(catalog_rvs, measured_rvs, 'k.', label="Night {0} data".format(n))
    # Now fit a line to them.
    try:
        linefit, cov = np.polyfit(catalog_rvs, measured_rvs, 1, cov=True)
    except TypeError:
        if len(catalog_rvs) == 0:
            nightfit_results[i,:].mask = True
            displacements[n] = catalog_rvs
            continue
    nightfit_results[i,0:2] = linefit
    nightfit_results[i,2:4] = np.sqrt(np.diag(cov))
    reference_offset[n] = - linefit[1]
    linemodel = np.poly1d(linefit)
    plt.plot(catalog_rvs, measured_rvs+reference_offset[n], 'k.', label="Night {0} data".format(n))
    # Plot best-fit line and one-to-one line.
    minval = min(min(measured_rvs), min(catalog_rvs))
    maxval = max(max(measured_rvs), max(catalog_rvs))
    plt.plot([minval, maxval], [linemodel(minval)-linefit[1], linemodel(maxval)-linefit[1]], 'k--', label=str(linemodel))
    plt.plot([minval, maxval], [minval, maxval], 'r-', label="y=x")
    plt.xlabel("Catalog RV (km/s)")
    plt.ylabel("Measured RV - offset")
    plt.legend(loc="upper left")
    plt.title("Night {0} Corrected RV".format(n))
    
    displacements[n] = measured_rvs + reference_offset[n] - catalog_rvs
    
post_subtraction_errors = {k: np.std(v) for k, v in displacements.iteritems()}

/home/regulus/simonian/.conda/envs/iraf27/lib/python2.7/site-packages/numpy/core/_methods.py:135: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/home/regulus/simonian/.conda/envs/iraf27/lib/python2.7/site-packages/numpy/core/_methods.py:105: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/home/regulus/simonian/.conda/envs/iraf27/lib/python2.7/site-packages/numpy/core/_methods.py:127: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [541]:
# Now write the internight table.
nightfit_table = Table(nightfit_results, names=["Slope", "Intercept", "Slope Err", "Intercept Err"])
nightfit_table["Night"] = obsnights
nightfit_table["Scatter"] = [post_subtraction_errors[n] for n in nightfit_table["Night"]]
nightfit_table.sort("Night")
nightfit_table = nightfit_table["Night", "Slope", "Intercept", "Slope Err", "Intercept Err", "Scatter"]

nightfit_table.meta["comments"] = [
    "This file contains the standard-fit parameters for each night.",
    "",
    "The first column is the night number. The next four are the slope, intercept,",
    "slope error, and intercept error of the measured vs catalog values of the",
    "standards. The last column is the scatter of the standards with respect to the",
    "one-to-one line.",
    "",
    "The intercept column describes the displacement of the reference RV from the ",
    "catalog RVs. They are used to calibrate the reference RVs."
    "",
    "No standards were observed during Night 7, so it is left blank."
]

nightfit_table.write(os.path.join(IMAGE_PATH, CALIB_FOLDER, "RV_Internight_Params.txt"), format="ascii.fixed_width", 
                     overwrite=True)
    

In [58]:
# Plot the RV error of the standards throughout the night to see if twilight causes problems.
for n, disp in displacements.iteritems():
    plt.figure()
    if len(disp) > 0:
        plt.plot(obstimes[n]-int(obstimes[n][0]), displacements[n], 'bs')
    plt.xlabel("JD")
    plt.ylabel("RV Error")
    plt.title("Night {0} Standards Errors ".format(n))
    plt.savefig(os.path.join(BINARY_PATH, "plots", "Standard_Comparisons", "Night{0:d}_error_time.png".format(n)))

In [59]:
for n, coeffarray in coefficients.iteritems():
    plt.figure()
    offsets = coeffarray[:,1]
    center = np.mean(offsets)
    width = np.std(offsets)
    plt.hist(offsets, normed=True)
    edges = center + 3 * width * np.array([-1, 1])
    xvalues = np.linspace(edges[0], edges[1], 1000)
    distvalues = stats.norm.pdf(xvalues, center, width)
    plt.plot(xvalues, distvalues, '-', label="Width: {0:.2f}".format(width))
    plt.xlabel("Offset velocity (km/s)")
    plt.ylabel("Normalized density")
    plt.legend(loc="upper left")
    plt.title("Night {0} Offset Distribution".format(n))

In [23]:
# Standard RV curves will be a 3-tuple which contains the nights of the observation, the times of the observation, and 
# the rv values of the observations.
standard_rv_curves = {}
for n in obsnights:
    specfile = linearized_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, specfile)) as standards:
        for stand in itertools.islice(standards, 0, 1):
            refspec = stand[:-1]
    compact_reference = compact_standard(refspec)
    ref_basefile = target_cor_template.format(n, obj_types[0], compact_reference.upper())
    try:
        ref_rvs = open(os.path.join(IMAGE_PATH, CALIB_FOLDER, ref_basefile))
    except IOError:
        # Check if the night of the reference is equal to the current night.
        refnum = int(refspec[5:oldname.index(os.sep)])
        if refnum != n:
            continue
        else:
            raise
    with ref_rvs:
        # The reference spectrum is the first one in this case, so skip it.
        for rvfile in itertools.islice(ref_rvs, 1, None):
            try:
                vel_table = Table.read(
                    os.path.join(IMAGE_PATH, CALIB_FOLDER, rvfile[:-1]+".txt"), format="ascii.commented_header", 
                    fill_values=[("", "0"), ("INDEF", "0")], header_start=13, guess=False, 
                    names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 'TDR', 
                           'VOBS', 'VREL', 'VHELIO', 'VERR'])
            except IOError:
                # Skip cases where the target spectrum is the reference spectrum.
                # rvfile:  night1/night1.c072n1c072
                # refspec: night1/night1.c072.ms.fits
                num_start = rvfile.index("c")+1
                num_end = num_start+3
                rvnum = rvfile[num_start:num_end]+r
                refnum = refspec[num_start:num_end]
                if rvnum == refnum:
                    continue
                else:
                    raise
            
            rv = vel_table["VHELIO"][-1]+reference_offset[n]
            time = vel_table["HJD"][-1]
            objname = vel_table["OBJECT"][-1]
            if objname == "HD183743":
                print rvfile
            
            nights, times, rvs = standard_rv_curves.setdefault(objname, ([], [], []))
            nights.append(n)
            times.append(time)
            rvs.append(rv)
            
standard_rv_curves = {k: (v[0], np.array(v[1]), np.array(v[2])) for k,v in standard_rv_curves.iteritems()}

In [345]:
colors = {0: (0/255.0, 0/255.0, 0/255.0), 1: (255/255.0, 109/255.0, 182/255.0), 2: (0/255.0, 109/255.0, 219/255.0), 
          3: (182/255.0, 219/255.0, 255/255.0), 4: (219/255.0, 209/255.0, 0/255.0)}
for i, (stand, (nights, times, rvs)) in itertools.islice(enumerate(standard_rv_curves.iteritems()), 0, None):
    modstand = i % 5
    if modstand == 0:
        plt.figure()
        plt.xlabel("Observation Time (HJD)")
        plt.ylabel("RV (km/s)")
    xvals = times-7904.0
    yvals = rvs
    yerrs = np.array([post_subtraction_errors[n] for n in nights])
    plt.errorbar(xvals, yvals, yerrs, marker="o", color=colors[modstand])
    edgejd = np.array([0, 14])
    standrvs = np.array([rv_lookup[stand], rv_lookup[stand]])
    plt.plot(edgejd, standrvs, linestyle='-', color=colors[modstand], label=stand)
    plt.legend(loc="lower left")

In [38]:
for stand, (nights, obstimes, rvcurve) in itertools.islice(standard_rv_curves.iteritems(), 0, None):
    plt.figure()
    rverr = 12
    basetime = 7904 # The base in HJD for some reason.
    times = obstimes - basetime #% rotperiod
    catrv = rv_lookup[stand]
    plt.errorbar(times, rvcurve-catrv, yerr=rverr/2, fmt='ko')
    plt.plot([0, 14], [0, 0], 'k--')
    plt.xlabel("Time (HJD)")
    plt.ylabel("Heliocentric RV Offset (km/s)")
    plt.title(stand)
    plt.xlim(0, 15)
    plt.ylim(-50, 50)
#    plt.savefig(os.path.join(BINARY_PATH, "plots", "Kepler_RVs", keplerobj+".png"))

In [61]:
# Plot full residuals
standard_residuals = {k: standard_rv_curves[k][2] - rv_lookup[k] for k in standard_rv_curves.keys()}
all_residuals = np.concatenate(standard_residuals.values())
plt.hist(all_residuals, bins=20, normed=True, 
         label="Mean: {0:.2f}; Disp: {1:.2f}".format(np.mean(all_residuals), np.std(all_residuals)))
xvals = np.linspace(-40, 40, 100)
expected_fit = stats.norm(loc=0, scale=10).pdf(xvals)
plt.plot(xvals, expected_fit, linestyle="-", color="orange", label="Mean: 0; Disp: 10")
plt.xlabel("Residual (km/s)")
plt.ylabel("Offset density")
plt.title("Distribution of {0} observations over {1} standards".format(len(all_residuals), len(standard_residuals)))
plt.legend(loc="upper right")

In [72]:
# Split residuals into length groups. And then plot sample mean and distribution 
obs_lengths = collections.defaultdict(list)
for stand, res in standard_residuals.iteritems():
    obs_lengths[len(res)].append(stand)
obs_gtone_lengths = {k: v for k,v in obs_lengths.iteritems() if k > 1}

In [24]:
def rebin( a, newshape ):
        '''Rebin an array to a new shape.
        '''
        assert len(a.shape) == len(newshape)

        slices = [ slice(0,old, float(old)/new) for old,new in zip(a.shape,newshape) ]
        coordinates = mgrid[slices]
        indices = coordinates.astype('i')   #choose the biggest smaller integer index
        return a[tuple(indices)]

In [74]:
xvals = np.linspace(-30, 30, 100)
means = np.array([np.mean(res) for res in standard_residuals.values()])
his, bins, patches = plt.hist(means, bins=30, range=(xvals[0], xvals[-1]),
                              label="Mean: {0:.2f}; Disp: {1:.2f}".format(np.mean(means), np.std(means)))
scales = 11.6/np.sqrt(obs_lengths.keys())[:,np.newaxis]
dists = stats.norm(loc=0, scale=scales).pdf(xvals[np.newaxis,:])
occurrences = np.array(map(len, obs_lengths.values()), dtype=np.float)
expected_fit = np.sum(dists * occurrences[:,np.newaxis], axis=0) / len(standard_residuals)
norm = (xvals[1]-xvals[0])*(expected_fit[0]/2.0 + np.sum(expected_fit[1:-1]) + expected_fit[-1]/2.0)
mean = (xvals[1]-xvals[0])*(
    expected_fit[0]*xvals[0]/2.0 + np.sum(expected_fit[1:-1]*xvals[1:-1]) + expected_fit[-1]*xvals[-1]/2.0)/norm
disp = np.sqrt(((xvals[1]-xvals[0])*(
    expected_fit[0]*xvals[0]**2/2.0 + np.sum(expected_fit[1:-1]*xvals[1:-1]**2) + expected_fit[-1]*xvals[-1]**2/2.0)/norm) - mean**2)
#plt.plot(xvals, expected_fit*len(means)*(bins[1]-bins[0]), linestyle='-', color="orange", 
#         label="Mean: {0:.1f}; Disp: {1:.1f}".format(mean, disp))
binned_fit = np.diff(np.interp(bins, xvals, np.cumsum(expected_fit)))
plt.step(bins[:-1], binned_fit*len(means)*(xvals[1]-xvals[0]), where="post", color="orange",
         label="Mean: {0:.1f}; Disp: {1:.1f}".format(mean, disp))
print (xvals[1]-xvals[0])*(binned_fit[0]/2.0 + np.sum(binned_fit[1:-1]) + binned_fit[-1]/2.0)
plt.legend(loc="upper right")
plt.xlabel("Offset of Mean RV from Catalog")
plt.ylabel("Number of RVs in mean RV bin")
plt.title("Distribution of sample means")

0.999242371062


In [80]:
xvals = np.linspace(0, 20, 100)
standard_stds = np.array([np.std(res) for res in standard_residuals.values() if len(res) > 1])
nobs = np.array([len(res) for res in standard_residuals.values() if len(res) > 1])[:,np.newaxis]
his, bins, patches = plt.hist(standard_stds, bins=10, range=(xvals[0], xvals[-1]), 
                              label="Mean: {0:.2f}; Disp: {1:.2f}".format(np.mean(standard_stds), np.std(standard_stds)))
N = nobs
exp_err = 11.6
expected_fit = np.mean(2 * (N/(2*exp_err**2))**((N-1)/2.0) / (gamma((N-1)/2.0)) * np.exp(-N*xvals[np.newaxis,:]**2/(2*exp_err**2))*xvals[np.newaxis,:]**(N-2), axis=0)
norm = (xvals[1]-xvals[0])*(expected_fit[0]/2.0 + np.sum(expected_fit[1:-1]) + expected_fit[-1]/2.0)
mean = (xvals[1]-xvals[0])*(
    expected_fit[0]*xvals[0]/2.0 + np.sum(expected_fit[1:-1]*xvals[1:-1]) + expected_fit[-1]*xvals[-1]/2.0)/norm
disp = np.sqrt(((xvals[1]-xvals[0])*(
    expected_fit[0]*xvals[0]**2/2.0 + np.sum(expected_fit[1:-1]*xvals[1:-1]**2) + expected_fit[-1]*xvals[-1]**2/2.0)/norm) - mean**2)
binned_fit = np.diff(np.interp(bins, xvals, np.cumsum(expected_fit)))
plt.step(bins[:-1], binned_fit*len(means)*(xvals[1]-xvals[0]), where="post", color="orange",
         label="Mean: {0:.1f}; Disp: {1:.1f}".format(mean, disp))
plt.legend(loc="upper right")
plt.xlabel("Scatter of RV curves")
plt.ylabel("Number of RVs in scatter bin")
plt.title("Distribution of RV scatters")

In [25]:
# Let's also do chi-squared tests.
# These standards should follow the null hypothesis of being RV nonvariable.
rv_err = 12
xvals = np.linspace(0.01, 21, 1000)
# Get the distribution of empirical chi-squareds.
chi_sqs = []
dofs = []
for stand, (nights, obstimes, rvcurve) in itertools.islice(standard_rv_curves.iteritems(), 0, None):
    chi_sq = np.sum((rvcurve - rv_lookup[stand])**2)/rv_err**2
    chi_sqs.append(chi_sq)
    dofs.append(len(rvcurve))
chi_sqs = np.array(chi_sqs)
dofs = np.array(dofs)
his, bins, patches = plt.hist(chi_sqs, bins=7, range=(xvals[0], xvals[-1]), 
                              label="RV Nonvar")
# Now predict the distribution of chi-squared expected.
pred_chisq = stats.chi2.pdf(xvals[:,np.newaxis], dofs)
full_chisq = np.mean(pred_chisq, axis=1)
binned_fit = np.diff(np.interp(bins, xvals, np.cumsum(full_chisq)))
plt.step(bins[:-1], binned_fit*len(chi_sqs)*(xvals[1]-xvals[0]), where="post", color="orange",
         label="Predicted Chi-sq")
plt.xlabel("Chi-sq")
plt.ylabel("Number of Standards in Bin")
plt.legend(loc="upper right")
plt.title("Standard Chi-Sq distribution")

In [204]:
print his
print binned_fit
print np.sum((his-binned_fit)**2/binned_fit**2)
print stats.chisqprob(12.6, 6)

[ 25.  17.  12.   8.   4.   2.   2.]
[ 11.19039172  12.23340906  10.25195768   6.63378613   3.66017082
   1.82099329   0.84167355]
3.65846726032
0.0498464931724


/home/regulus/simonian/.conda/envs/iraf27/lib/python2.7/site-packages/ipykernel_launcher.py:4: DeprecationWarning: `chisqprob` is deprecated!
stats.chisqprob is deprecated in scipy 0.17.0; use stats.distributions.chi2.sf instead.
  after removing the cwd from sys.path.


In [26]:
# Let's look at the anderson-darling test.
chisq_interp = interp1d(xvals, full_chisq)

[ 0.25828202  0.16113646  0.13258413  0.11777255  0.10843673  0.10192172
  0.09707989  0.09332428  0.09031976  0.08785941  0.08580777  0.08407202
  0.08258622  0.08130204  0.08018322  0.07920194  0.07833646  0.07756953
  0.07688728  0.0762784   0.07573357  0.07524505  0.07480634  0.07441194
  0.07405715  0.07373796  0.0734509   0.07319294  0.07296147  0.07275416
  0.07256898  0.07240414  0.07225803  0.07212922  0.07201643  0.07191852
  0.07183444  0.07176326  0.07170413  0.07165628  0.07161899  0.07159163
  0.0715736   0.07156436  0.0715634   0.07157027  0.07158453  0.07160579
  0.07163368  0.07166785  0.07170798  0.07175378  0.07180495  0.07186124
  0.0719224   0.07198819  0.0720584   0.07213281  0.07221123  0.07229347
  0.07237935  0.07246871  0.07256138  0.07265721  0.07275605  0.07285777
  0.07296223  0.07306931  0.07317887  0.07329081  0.073405    0.07352135
  0.07363973  0.07376006  0.07388224  0.07400617  0.07413175  0.0742589
  0.07438754  0.07451758  0.07464894  0.07478155  0.

## Spectral variation

In [139]:
for n, disp in displacements.iteritems():
    if n == 7:
        continue
    plt.figure()
    templatefile = reference_spectra[n]
    templatename =  iraf.hedit(templatefile, "OBJECT", ".", Stdout=1)[0].split("=")[1].strip()
    standard_files = linearized_target_template.format(n, obj_types[0])
    standnames = []
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standard_files)) as standards:
        for stand in standards:
            if stand[:-1] != templatefile:
                standnames.append(iraf.hedit(stand[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip())
    standjks = np.array([JK_lookup[s] for s in standnames])
    standdiffs = standjks - JK_lookup[templatename]
    plt.plot(standdiffs, disp[1:], 'bs')
    # Now fit a line to it
    linefit, cov = np.polyfit(standdiffs, disp[1:], 1, cov=True)
    slope_err = np.sqrt(cov[0,0])
    intercept_err = np.sqrt(cov[1,1])
    linemodel = np.poly1d(linefit)
    plt.plot([min(standdiffs), max(standdiffs)], [linemodel(min(standdiffs)), linemodel(max(standdiffs))], 'k-')
    correlation = stats.pearsonr(standdiffs, disp[1:])
    print "Night {0:d} Correlation Coefficient: {1:.2f}".format(n, correlation[0])
    plt.xlabel("J-K difference")
    plt.ylabel("RV error")
    plt.title("Night {0} Template Mismatch".format(n))
    

Night 1 Correlation Coefficient: -0.12
Night 3 Correlation Coefficient: 0.25
Night 4 Correlation Coefficient: 0.32
Night 5 Correlation Coefficient: 0.30
Night 6 Correlation Coefficient: 0.08
Night 8 Correlation Coefficient: 0.18
Night 9 Correlation Coefficient: -0.37
Night 10 Correlation Coefficient: 0.06
Night 11 Correlation Coefficient: -0.15
Night 12 Correlation Coefficient: 0.11
Night 13 Correlation Coefficient: 0.23
Night 14 Correlation Coefficient: 0.06


In [138]:
correlation

(-0.11738103424012108, 0.74673770592147748)


## Duplicate analysis (cross-correlation error)

In [202]:
# Get flexure of duplicate images
duplicate_filelist = "Duplicate_Standards.txt"                    
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, duplicate_filelist), "r") as dupnames:
    duplicate_fullfile = "Duplicate_Filenames.txt"
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, duplicate_fullfile), 'w') as dupfiles:
        for dup in dupnames:
            twofiles = dup.split()
            for singlefile in twofiles:
                dupfiles.write(singlefile+"\n")
extracted_target_filelist = "Duplicate_Extracted.txt"
# Make the filenames for the extracted objects
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, duplicate_fullfile), 'r') as oldfile, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist), 'w') as newfile:
                for oldname in oldfile:
                    newname = oldname.replace(".fit", ".ms.fits")
                    newfile.write(newname)
iraf.apall("@"+duplicate_fullfile, output="@"+extracted_target_filelist, intera="no")

Oct 17 13:30: EXTRACT - Output spectrum night1/night1.075.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night1/night1.076.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night1/night1.113.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night1/night1.114.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night1/night1.122.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night1/night1.123.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night1/night1.127.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night1/night1.128.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night1/night1.129.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night1/night1.130.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night3/night3.091.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night3/night3.092.ms already exists
Oct 17 13:30: EXTRACT - Output spectrum night3/night3.116.ms already exists
Oct 17 13:30

In [203]:
for spec in arctypes:
    extracted_calib_filelist = "Duplicate_{0}.txt".format(spec.capitalize())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist), "r") as oldfile,\
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), "w") as newfile:
            for oldname in oldfile:
                nightname = oldname[:oldname.index(os.sep)]+"."
                newname = oldname.replace(nightname, nightname+spec)
                newfile.write(newname)
                try:
                    os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                    print "Removed " + newname
                except OSError:
                    pass
    apall_repeat_file = "Duplicate_Repeat_{0}.txt".format(spec.capitalize())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), "r") as oldfile,\
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, apall_repeat_file), "w") as newfile:
            for oldname in oldfile:
                nightname = oldname[:oldname.index(os.sep)]
                newname = os.path.join("Calibrations", "{0}_{1}.fit\n".format(nightname.capitalize(), spec.capitalize()))
                newfile.write(newname)
    iraf.apall("@"+apall_repeat_file, out="@"+extracted_calib_filelist, ref="@"+duplicate_fullfile, recen=False, 
               trace=False, back="none", intera=False)
    print "Is this being done?"
    print "Things Extracted."
    subtracted_calib_filelist = "Duplicate_{0}_Subtracted.txt".format(spec.capitalize())
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), "r") as oldfile,\
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, subtracted_calib_filelist), "w") as newfile:
            for oldname in oldfile:
                newname = oldname.replace(spec, "s"+spec)
                newfile.write(newname)
                try:
                    os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
                    print "Removed " + newname
                except OSError:
                    pass
    iraf.continuum.func = "chebyshev"
    iraf.continuum.order = 15
    iraf.continuum.high_rej = 3
    iraf.continuum.low_rej = 0
    try:
        iraf.continuum("@"+extracted_calib_filelist, "@"+subtracted_calib_filelist, intera="no")
    except iraf.IrafError:
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_calib_filelist), "r") as infile:
            contents = infile.readlines()
            if not contents:
                pass
            else:
                raise
                
    iraf.reidentify(os.path.join("calib_test", "{0}spec".format(spec)), "@"+subtracted_calib_filelist, intera="no")
    
fullspec_filelist = "Duplicate_{0}_Fullspec.txt".format(spec)
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, "Duplicate_Ar_Subtracted.txt"), "r") as oldfile,\
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist), "w") as newfile:
        for oldname in oldfile:
            newname = oldname.replace("sar", "full")
            shutil.copy(os.path.join(IMAGE_PATH, CALIB_FOLDER, oldname[:-1]),
                        os.path.join(IMAGE_PATH, CALIB_FOLDER, newname[:-1]))
            newfile.write(newname)
full_spec_table = []
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist), "r") as fullspecs:
    for fullimg in fullspecs:
        fulldb, ext = os.path.splitext(os.path.join("database", "id"+fullimg))
        for spec in ["ne", "ar", "xe"]:
            specimg = fullimg.replace("full", "s"+spec)
            specdb, ext = os.path.splitext(os.path.join("database", "id"+specimg))
            with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, specdb)) as specdata:
                spec_fullfile = specdata.read()
            spec_features = spec_fullfile[spec_fullfile.rindex("begin"):]
            spec_length_line_start = spec_features.index("features")
            spec_length_line_end = spec_features.index("\n", spec_length_line_start)
            spec_numlines = int(spec_features[spec_length_line_start:spec_length_line_end].split("\t")[1])
            spec_table_start = spec_length_line_end+1
            spec_table_end = spec_features.index("function")-2
            spec_table = spec_features[spec_table_start:spec_table_end].split("\n")
            full_spec_table = full_spec_table + spec_table
        full_numlines = len(full_spec_table)
        full_feat_table = Table.read(full_spec_table, format="ascii.fixed_width_no_header", 
                                     names=("Pixel", "Fit", "User", "Fwidth", "Wt", "dunno"), 
                                     col_starts=(0, 15, 26, 37, 43, 45), col_ends=(14, 25, 36, 40, 44, 46))
        full_feat_table["Count"] = np.arange(len(full_feat_table))
        full_feat_table.sort("Pixel")
        sorted_table = [full_spec_table[i] for i in full_feat_table["Count"]]
        new_feature_table = "\n".join(sorted_table)

        # Now let's piece together the new file. First make the time comment.
        a = datetime.now()
        comment_line = "# " + a.strftime("%a %H:%M:%S %d-%b-%Y") + "\n"
         # Then make the header:
        spec_head_start = 0
        # Note that this includes the leading tab character in the header, not as part of the "feature" line.
        spec_head_end = spec_length_line_start
        spec_header = spec_features[spec_head_start:spec_head_end]
        full_header = spec_header.replace("s"+spec, "full")
        # Now the feature line will be added on.
        full_feature_line = "features\t{0:d}\n".format(full_numlines)
        # Lastly we want the footer, which doesn't actually contain any useful information, but we will include.
        footer = spec_features[spec_table_end:]
        # Now add them all together!
        fullfile = comment_line + full_header + full_feature_line + new_feature_table + footer
        
        # Write the result to a file.
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fulldb), 'a') as fulldata:
            fulldata.write(fullfile)

Removed night1/night1.ne075.ms.fits

Removed night1/night1.ne076.ms.fits

Removed night1/night1.ne113.ms.fits

Removed night1/night1.ne114.ms.fits

Removed night1/night1.ne122.ms.fits

Removed night1/night1.ne123.ms.fits

Removed night1/night1.ne127.ms.fits

Removed night1/night1.ne128.ms.fits

Removed night1/night1.ne129.ms.fits

Removed night1/night1.ne130.ms.fits

Removed night3/night3.ne091.ms.fits

Removed night3/night3.ne092.ms.fits

Removed night3/night3.ne116.ms.fits

Removed night3/night3.ne117.ms.fits

Removed night4/night4.ne110.ms.fits

Removed night4/night4.ne111.ms.fits

Removed night4/night4.ne205.ms.fits

Removed night4/night4.ne206.ms.fits

Removed night5/night5.ne082.ms.fits

Removed night5/night5.ne083.ms.fits

Removed night6/night6.ne108.ms.fits

Removed night6/night6.ne109.ms.fits

Removed night6/night6.ne129.ms.fits

Removed night6/night6.ne130.ms.fits

Removed night6/night6.ne154.ms.fits

Removed night6/night6.ne155.ms.fits

Removed night6/night6.ne167.ms.fits

R

In [204]:
calibrated_target_filelist = "Duplicates_Calib.txt"
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist), "r") as oldfile, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_target_filelist), "w") as newfile:
        for oldname in oldfile:
            nightstr = oldname[:oldname.index(os.sep)] + "."
            newname = oldname.replace(nightstr, nightstr+"c")
            newfile.write(newname)
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, fullspec_filelist)) as fullspec, \
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, extracted_target_filelist)) as stand:
        for ref, obj in zip(fullspec, stand):
            print obj[:-1]
            iraf.hedit(obj[:-1], "REFSPEC1", ref[:-1], add=True, verify=False)
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_target_filelist)) as caltarg:
    for targ in caltarg:
        try:
            os.remove(os.path.join(IMAGE_PATH, CALIB_FOLDER, targ[:-1]))
        except OSError:
            pass
iraf.dispcor("@"+extracted_target_filelist, "@"+calibrated_target_filelist, linearize=True)

night1/night1.075.ms.fits
night1/night1.075.ms.fits,REFSPEC1: night1/night1.full075.ms.fits -> night1/night1.full075.ms.fits
night1/night1.075.ms.fits updated
night1/night1.076.ms.fits
night1/night1.076.ms.fits,REFSPEC1: night1/night1.full076.ms.fits -> night1/night1.full076.ms.fits
night1/night1.076.ms.fits updated
night1/night1.113.ms.fits
night1/night1.113.ms.fits,REFSPEC1: night1/night1.full113.ms.fits -> night1/night1.full113.ms.fits
night1/night1.113.ms.fits updated
night1/night1.114.ms.fits
night1/night1.114.ms.fits,REFSPEC1: night1/night1.full114.ms.fits -> night1/night1.full114.ms.fits
night1/night1.114.ms.fits updated
night1/night1.122.ms.fits
night1/night1.122.ms.fits,REFSPEC1: night1/night1.full122.ms.fits -> night1/night1.full122.ms.fits
night1/night1.122.ms.fits updated
night1/night1.123.ms.fits
night1/night1.123.ms.fits,REFSPEC1: night1/night1.full123.ms.fits -> night1/night1.full123.ms.fits
night1/night1.123.ms.fits updated
night1/night1.127.ms.fits
night1/night1.127.ms

night6/night6.109.ms.fits: REFSPEC1 = 'night6/night6.full109.ms.fits 1.'
night6/night6.c109.ms.fits: ap = 1, w1 =  4346.29, w2 = 6063.294, dw = 1.010597, nw = 1700
night6/night6.129.ms.fits: REFSPEC1 = 'night6/night6.full129.ms.fits 1.'
night6/night6.c129.ms.fits: ap = 1, w1 = 4346.291, w2 = 6063.294, dw = 1.010597, nw = 1700
night6/night6.130.ms.fits: REFSPEC1 = 'night6/night6.full130.ms.fits 1.'
night6/night6.c130.ms.fits: ap = 1, w1 = 4346.287, w2 = 6063.294, dw = 1.010598, nw = 1700
night6/night6.154.ms.fits: REFSPEC1 = 'night6/night6.full154.ms.fits 1.'
night6/night6.c154.ms.fits: ap = 1, w1 = 4346.302, w2 = 6063.279, dw = 1.010581, nw = 1700
night6/night6.155.ms.fits: REFSPEC1 = 'night6/night6.full155.ms.fits 1.'
night6/night6.c155.ms.fits: ap = 1, w1 = 4346.296, w2 = 6063.284, dw = 1.010588, nw = 1700
night6/night6.167.ms.fits: REFSPEC1 = 'night6/night6.full167.ms.fits 1.'
night6/night6.c167.ms.fits: ap = 1, w1 = 4346.292, w2 = 6063.293, dw = 1.010595, nw = 1700
night6/night6.16

In [205]:
# These are all really cool!
def grouper(n, iterable):
    "s -> (s0,s1,...sn-1), (sn,sn+1,...s2n-1), (s2n,s2n+1,...s3n-1), ..."
    return itertools.izip(*[iter(iterable)]*n)

pairwise = functools.partial(grouper, 2)

In [222]:
# Now get every other entry from the calibrated targets.
duplicate_fxcor_output = "Duplicates_Pair_FXcor.txt"
iraf.filtpars.cuton = 15
iraf.filtpars.cutoff = 150
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_target_filelist), "r") as oldfile,\
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, duplicate_fxcor_output), "w") as newfile:
        for old1, old2 in pairwise(oldfile):
            compact = compact_standard(old2)
            num1 = old1[-12:-9]
            print num1
            newname = os.path.splitext(os.path.splitext(old1)[0])[0].replace("c"+num1, num1+"cc"+compact)
            newfile.write(newname+"\n")
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, calibrated_target_filelist)) as calib_targets,\
     open(os.path.join(IMAGE_PATH, CALIB_FOLDER, duplicate_fxcor_output)) as output_names:
        for ((targ, temp), output) in itertools.izip(pairwise(calib_targets), output_names):
            iraf.fxcor(targ[:-1], temp[:-1], out=output[:-1], intera="no")

075
113
122
127
129
091
116
110
205
082
108
129
154
167
172


In [71]:
shifts = []
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, duplicate_fxcor_output)) as shiftfiles:
    for shiftfile in shiftfiles:
        vel_table = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, shiftfile[:-1]+".txt"), 
                               format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], header_start=13, 
                               guess=False, names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 
                                                   'FWHM', 'TDR', 'VOBS', 'VREL', 'VHELIO', 'VERR'])
        shift = vel_table["VHELIO"][-1]
        shifts.append(shift)
shiftarray = np.array(shifts)

NameError: name 'duplicate_fxcor_output' is not defined

In [70]:
plt.hist(shiftarray, bins=6, normed=True)
meanoffset = np.mean(shiftarray)
stdoffset = np.std(shiftarray)
normal = stats.norm.pdf(np.linspace(-25, 25, 100), loc=meanoffset, scale=stdoffset)
plt.plot(np.linspace(-30, 30, 100), normal)
plt.xlabel("Velocity offset (km/s)")
plt.ylabel("Offset distribution")
plt.title("Velocity offsets from adjacent exposures")
print(meanoffset)
print(stdoffset)
print(stdoffset/np.sqrt(len(shiftarray)))

NameError: name 'shiftarray' is not defined

In [221]:
plt.hist(shiftarray, histtype="step", cumulative=True, normed=True, align="right", bins=100)
plt.plot(np.linspace(-30, 30, 100), np.cumsum(normal)/2)
plt.xlabel("Velocity offset (km/s)")
plt.ylabel("Cumulative distribution")
plt.title("Empirical distribution function for adjacent exposure velocity offsets")

In [55]:
calibrated_target_filelist = "Duplicates_Calib.txt"

## Flexure check

In [9]:
standard_skyline = {
    12: np.array([
        489.799, 489.530, 489.245, 489.827, 489.817, 489.775, 489.769, 489.634, 489.623, 489.640,
        489.769, 489.730, 489.913, 489.972, 489.746, 489.612, 489.668, 489.517, 489.516, 489.488,
        489.431, 489.307, 489.222, 489.116, 489.064, 489.107, 489.058, 489.099, 488.852, 488.894, 
        488.709, 488.761, 488.875, 489.327, 489.236, 489.005, 489.142, 489.043, 489.849, 488.653,
        488.550, 488.801, 488.980, 488.731, 488.465, 488.319, 488.156, 487.662, 488.127, 488.584
        ]),
    4: np.array([
        489.598, 489.567, 489.955, 489.975, 489.844, 489.557, 489.747, 489.442, 489.309, 489.361,
        489.226, 489.243, 489.212, 489.230, 489.153, 489.516, 489.405, 489.441, 489.245, 489.208,
        489.145, 489.026, 488.977, 489.044, 488.948, 488.895, 488.850, 488.955, 488.821, 488.781, 
        489.571, 489.803, 489.678, 489.076, 489.128, 489.477, 488.248, 488.205, 487.979, 489.692,
        489.791, 489.099, 488.911, 488.302, 486.658
        ])}

kic_skyline = {
    12: np.array([
        488.697, 488.658, 488.665, 488.649, 488.636, 488.565, 488.678, 488.587, 488.554, 488.583,
        488.529, 488.488, 488.469, 488.495, 488.493, 488.485, 488.445, 488.446, 488.504, 488.438,
        488.389, 488.367, 488.339, 488.295, 488.378, 488.304, 488.343, 488.294, 488.398, 488.334,
        488.346, 488.303, 488.402, 488.339, 488.372, 488.516, 488.528, 488.510, 488.474, 488.496,
        488.497, 488.404, 488.524, 488.506, 488.510, 488.456, 488.531, 488.662, 488.576, 488.715,
        488.725, 488.893
        ]),
    4: np.array([
        488.797, 488.829, 488.851, 488.803, 488.820, 488.833, 488.785, 488.795, 488.770, 488.746,
        488.735, 488.748, 488.731, 488.748, 488.740, 488.764, 488.781, 488.766, 488.772, 488.761,
        488.823, 488.795, 488.825, 488.844, 488.804, 488.954, 488.928, 488.854, 488.929, 488.899,
        488.944, 489.009, 488.917, 488.919, 489.045, 488.954, 489.104, 489.219, 489.194, 489.222,
        489.270, 489.235, 489.404
    ])
}

In [14]:
# Make a dictionary that points arcs to the corresponding target file.
# This basically goes through each arc filename, and finds the corresponding object filename.
arc_lookup = {}
for n in obsnights:
    targnames = []
    for targ in obj_types:
        arcfiles = arc_template.format(n, targ)
        with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, arcfiles)) as arcs:
            for arc in arcs:
                arcname = iraf.hedit(arc[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip()
                # For multi-word keyword values, hedit embeds it in quotes.
                objname = arcname.strip('"')[:-4]
                # Use the existing file lookup to get the filename of the corresponding object.
                # We only want objects that were observed in the same night.
                objfiles = filter(lambda x: "night{0:d}".format(n) in x, file_lookup[objname])
                # If the standard was observed exactly once in the night, then there should be
                # exactly one standard observation that night.
                # If the standard was observed multiple times, then keep track of which exposure
                # this corresponds to. Hopefully, the number of corresponding objects and arcs
                # are equal.
                repeat_dict = {}
                if len(objfiles) == 1:
                    objfile = objfiles[0]
                else:
                    objcount = repeat_dict.get(objname, 0)
                    objfile = objfiles[objcount]
                    repeat_dict[objname] = objcount + 1
                # Objfile will be in the form night12/night12.l136.ms.fits
                # When dealing with arcs, we don't care for the calibrated spectrum, so get rid of the l.
                arc_lookup[arc[:-1]] = objfile.replace("l", "")

In [322]:
# Collect the flexure value associated with each arc.
night=4
standard_skyline_pixels = standard_skyline[night]
kic_skyline_pixels = kic_skyline[night]

fxcor_pixels = collections.defaultdict(list)
hourangles = collections.defaultdict(list)
decs = collections.defaultdict(list)
jds = collections.defaultdict(list)
    
# Standard flex info.
flexure_files = fxcor_flexure_template.format(night, obj_types[0])
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, flexure_files)) as flex:
    for shiftfile in flex:
        shift_table = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, shiftfile[:-1]+".txt"), 
                                 format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                                 header_start=13, guess=False, 
                                 names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 
                                        'TDR', 'VOBS', 'VREL', 'VHELIO', 'VERR'])
        fxcor_pixels[obj_types[0]].append(shift_table["SHIFT"][-1])
        hourangles[obj_types[0]].append(iraf.hedit(shift_table["IMAGE"][-1], "HA", ".", Stdout=1)[0].split("=")[1].strip())
        decs[obj_types[0]].append(iraf.hedit(shift_table["IMAGE"][-1], "DEC", ".", Stdout=1)[0].split("=")[1].strip())
        jds[obj_types[0]].append(iraf.hedit(shift_table["IMAGE"][-1], "JD", ".", Stdout=1)[0].split("=")[1].strip())
        
# KIC Object flex info
# First, get the flexure for all the arcs.
# At the same time, get the skyline associated with the arc.
kic_arc_skylines = []
kic_filename = extracted_target_template.format(night, obj_types[1])
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_filename)) as kicobjs:
    kicfiles = kicobjs.readlines()
assert len(kicfiles) == len(kic_skyline_pixels)
kic_arcs = arc_template.format(night, obj_types[1])
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_arcs)) as arcs:
    for arc in arcs:
        hourangles[obj_types[1]].append(iraf.hedit(arc[:-1], "HA", ".", Stdout=1)[0].split("=")[1].strip())
        decs[obj_types[1]].append(iraf.hedit(arc[:-1], "DEC", ".", Stdout=1)[0].split("=")[1].strip())
        jds[obj_types[1]].append(iraf.hedit(arc[:-1], "JD", ".", Stdout=1)[0].split("=")[1].strip())
        # Now get the shift of the arc.
        arc_obj_file = arc_lookup[arc[:-1]]
        # The arc file should be of the form: night12/night12.134.fit\n
        # The object file should be of the form: night12/night12.135.ms.fits
        # And I want a file of the form: night12/night12.134t135.txt
        arcnum = arc[-8:-5]
        objnum = arc_obj_file[-11:-8]
        arc_shift_file = arc[:-1].replace(".fit", "t{0}.txt".format(objnum))
        
        shift_table = Table.read(os.path.join(IMAGE_PATH, CALIB_FOLDER, arc_shift_file), 
                                 format="ascii.commented_header", fill_values=[("", "0"), ("INDEF", "0")], 
                                 header_start=13, guess=False, 
                                 names=['OBJECT', 'IMAGE', 'REF', 'HJD', 'AP', 'CODES', 'SHIFT', 'HGHT', 'FWHM', 
                                        'TDR', 'VOBS', 'VREL', 'VHELIO', 'VERR'])
        fxcor_pixels[obj_types[1]].append(shift_table["SHIFT"][-1])
        # Add the corresponding skyline entry to kic_arc_skylines
        kic_arc_skylines.append(kic_skyline_pixels[kicfiles.index(arc_obj_file+"\n")])
    
hourangles = {x: Angle(hourangles[x], unit=u.hourangle) for x in hourangles}
decs = {x: Angle(decs[x], unit=u.degree) for x in decs}
jds = {x: np.array(jds[x], dtype=float) for x in jds}
kic_arc_skylines = np.array(kic_arc_skylines)

kic_jds = []
kic_files = linearized_target_template.format(night, obj_types[1])
with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, kic_files)) as kics:
    for kic in kics:
        kic_jds.append(iraf.hedit(kic[:-1], "JD", ".", Stdout=1)[0].split("=")[1].strip())
kic_jds = np.array(kic_jds, dtype=float)

In [306]:
kicfiles

['night4/night4.l141.ms.fits\n',
 'night4/night4.l142.ms.fits\n',
 'night4/night4.l143.ms.fits\n',
 'night4/night4.l144.ms.fits\n',
 'night4/night4.l145.ms.fits\n',
 'night4/night4.l147.ms.fits\n',
 'night4/night4.l148.ms.fits\n',
 'night4/night4.l149.ms.fits\n',
 'night4/night4.l150.ms.fits\n',
 'night4/night4.l151.ms.fits\n',
 'night4/night4.l153.ms.fits\n',
 'night4/night4.l154.ms.fits\n',
 'night4/night4.l155.ms.fits\n',
 'night4/night4.l156.ms.fits\n',
 'night4/night4.l157.ms.fits\n',
 'night4/night4.l158.ms.fits\n',
 'night4/night4.l160.ms.fits\n',
 'night4/night4.l161.ms.fits\n',
 'night4/night4.l162.ms.fits\n',
 'night4/night4.l163.ms.fits\n',
 'night4/night4.l164.ms.fits\n',
 'night4/night4.l165.ms.fits\n',
 'night4/night4.l167.ms.fits\n',
 'night4/night4.l168.ms.fits\n',
 'night4/night4.l169.ms.fits\n',
 'night4/night4.l170.ms.fits\n',
 'night4/night4.l171.ms.fits\n',
 'night4/night4.l172.ms.fits\n',
 'night4/night4.l173.ms.fits\n',
 'night4/night4.l175.ms.fits\n',
 'night4/n

In [351]:
zerotime = jds[obj_types[0]][0]
skyline_normalization = -(np.sum(fxcor_pixels[obj_types[0]]) + np.sum(fxcor_pixels[obj_types[1]]) - (
    np.sum(standard_skyline_pixels) + np.sum(kic_arc_skylines))) / (
    len(standard_skyline_pixels) + len(kic_arc_skylines))
print skyline_normalization
plt.plot((jds[obj_types[0]]-zerotime)*24, fxcor_pixels[obj_types[0]], 'ko', label="Arc Correlation")
plt.plot((jds[obj_types[0]]-zerotime)*24, standard_skyline_pixels-skyline_normalization, 'rx', label="Sky line")
plt.plot((jds[obj_types[1]]-zerotime)*24, fxcor_pixels[obj_types[1]], 'ko')
plt.plot((kic_jds-zerotime)*24, kic_skyline_pixels-skyline_normalization, 'rx')
plt.xlabel("Time (hour)")
plt.ylabel("Pixel shift")
plt.title("Flexure Measurements")
plt.legend(loc="lower left")

489.469226415


In [356]:
delt1 = fxcor_pixels[obj_types[0]] - standard_skyline_pixels
delt2 = fxcor_pixels[obj_types[1]] - kic_arc_skylines
zeropoint = (np.sum(delt1) + np.sum(delt2)) / (len(delt1) + len(delt2))
plt.plot((jds[obj_types[0]]-zerotime)*24, delt1-zeropoint, 'bs')
plt.plot((jds[obj_types[1]]-zerotime)*24, delt2-zeropoint, 'bs')
plt.xlabel("Time (hour)")
plt.ylabel("Residual (Cross-correlation - Skyline)")
plt.title("Flexure correction residuals")

In [355]:
# Fit a smooth function to the flexure
zerotime = jds[obj_types[0]][0]
relative_jd = (jds[obj_types[1]] - zerotime)*24
flexure_fit = np.polyfit(relative_jd, fxcor_pixels[obj_types[1]], 2)
flexfunc = np.poly1d(flexure_fit)
plt.plot(relative_jd, fxcor_pixels[obj_types[1]], 'ko', label="Arc flexure")
plt.plot(relative_jd, flexfunc(relative_jd) , 'k-', label="Arc flexure fit")
plt.plot((kic_jds-zerotime)*24, kic_skyline_pixels-skyline_normalization, 'rx', label="Skyline flexure")
flexure_uncert = np.std(kic_skyline_pixels-skyline_normalization-flexfunc((kic_jds-zerotime)*24))
plt.legend(loc="upper left")
plt.xlabel("Time since night began (hr)")
plt.ylabel("Flexure (px)")
plt.title("Night {0:d} Flexure".format(night))
print "Flexure RMS: {0:.2f} km/s".format(flexure_uncert*30)

Flexure RMS: 1.41 km/s


## Spectral Mismatch Variation

In [ ]:
# Read in the Standard information
standard_info = Table.read(os.path.join(BINARY_PATH, "Don_May_MDM_run", "Standard_SIMBAD.txt"), 
                            format="ascii.commented_header", header_start=0, data_start=4, data_end=-1, delimiter="|", 
                           fill_values=[("~", 0), ("", 0)], guess=False)
rv_lookup = dict(zip(standard_info["typed ident"], standard_info["radvel"]))
coord_lookup = dict(zip(standard_info["typed ident"], SkyCoord(standard_info["coord1 (ICRS,J2000/2000)"], 
                                                               unit=(u.hourangle, u.deg))))

In [120]:
# Select the median J-K star from each night.
standard_JK = standard_info["Mag J"] - standard_info["Mag K"]
JK_lookup = dict(zip(standard_info["typed ident"], standard_JK))
JK_references = {}
for n in obsnights:
    standardfile = linearized_target_template.format(n, obj_types[0])
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, standardfile)) as night_standards:
        standfiles = night_standards.readlines()
    standnames = []
    for stand in standfiles:
        standnames.append(iraf.hedit(stand[:-1], "OBJECT", ".", Stdout=1)[0].split("=")[1].strip())
    standjk = np.array([JK_lookup[s] for s in standnames])
    night_table = Table([standnames, standfiles, standjk], names=("Star", "File", "J-K"))
    night_table.sort("J-K")
    try:
        med_JK = night_table["File"][len(night_table)/2][:-1]
    except IndexError:
        continue
    JK_references[n] = med_JK

In [121]:
# Calculate offset for reference standards.
# This starts with cross-correlating to each other to determine what the measured RVs are.

iraf.fxcor.high_rej = 5
iraf.fxcor.low_rej = 2
iraf.fxcor.pixcor = "no"
iraf.fxcor.function = "gaussian"

iraf.continpars.order = 15
iraf.continpars.nitera = 10
iraf.continpars.grow = 1
iraf.continpars.c_inter = True

iraf.keywpars.ut = "TIME-OBS"
iraf.keywpars.epoch = "EQUINOX"

iraf.filtpars.cutoff = 250
iraf.filtpars.cuton = 25

# Just do standard correlation from night to night.
for n, template in itertools.islice(JK_references.iteritems(), 0, None):
    # Now begin going through standards for that night.
    template_cor_file = target_cor_template.format(n, obj_types[0], compact_standard(template).upper())
    target_list = linearized_target_template.format(n, obj_types[0])
    # Now populate template_cor_file
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_list), 'r') as oldfile, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file), "w") as newfile:
            for oldname in oldfile:
                newname = oldname.replace(".ms.fits", compact_standard(template))
                newfile.write(newname)
    # Now cross-correlate.
    with open(os.path.join(IMAGE_PATH, CALIB_FOLDER, target_list)) as targets, \
         open(os.path.join(IMAGE_PATH, CALIB_FOLDER, template_cor_file)) as outputs:
            for targ, out in zip(targets, outputs):
                if targ[:-1] != template:
                    iraf.fxcor(targ[:-1], template, out=out[:-1], interact="no")
                    print "Cross-correlating {0}".format(out[:-1])
                else:
                    print "Skipping {0}".format(out[:-1])

{1: 'night1/night1.ld05.ms.fits',
 3: 'night3/night3.l186.ms.fits',
 4: 'night4/night4.l214.ms.fits',
 5: 'night5/night5.l114.ms.fits',
 6: 'night6/night6.l163.ms.fits',
 8: 'night8/night8.l072.ms.fits',
 9: 'night9/night9.l115.ms.fits',
 10: 'night10/night10.l221.ms.fits',
 11: 'night11/night11.l093.ms.fits',
 12: 'night12/night12.l096.ms.fits',
 13: 'night13/night13.l090.ms.fits',
 14: 'night14/night14.l119.ms.fits'}